# Direct Residual Baseline Checks

This notebook implements the first residual-channel concept-discovery baseline for Residual SCBM.

The baseline question is simple:

> Do the learned residual dimensions already behave like discovered hidden concepts?

Protocol:

1. Load residual outputs from the split folders: `train/`, `val/`, `test/`.
2. Load true synthetic hidden residual labels from the matching dataset split.
3. Assert row alignment for every split.
4. Choose residual-to-hidden matches on validation only.
5. Report final recovery on test.

This is analogous to the concept-bank matching step in `cem-concept-discovery`, but here the residual channel is treated as one global missing-concept space instead of splitting each parent concept.


In [416]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, SparsePCA


import os
import re

## Select Experiment

Set `EXPERIMENT_PATH` to a Residual SCBM run that contains split-wise residual files:

```text
EXPERIMENT_PATH/train/res_mu.pt
EXPERIMENT_PATH/val/res_mu.pt
EXPERIMENT_PATH/test/res_mu.pt
```

The default below points at the hard synthetic run from the current analysis.


In [417]:
# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_1.0_beta_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_200_epochs_2026-06-04_17-42-45_5c587"
# )

# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/easy/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_easy_hid20_R20_dense_200_epochs_2026-06-07_19-24-48_e1bd1"
# )

# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/easy/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_easy_hid20_R50_sparse_200_epochs_2026-06-08_17-54-06_422d0"
# )


# EXPERIMENT_PATH = Path(
#     "experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_1.0_beta_1.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R50_sparse_200_epochs_2026-06-08_18-55-53_856ff"

# )

# EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/"
#     "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.0_hard_hid20_R20_hidden_strong_200_epochs_2026-06-10_13-12-25_84c8e"
    
# )


# EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/"
#                        "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_2026-06-10_13-13-13_9c899"
# )
       
EXPERIMENT_PATH = Path("experiments/scbm_residual/synthetic_res_scbm/hard/" 
                       "alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-10_16-22-14_09b78"
)            


CONCEPT_RESIDUAL_CHANNEL = True



assert EXPERIMENT_PATH.exists(), EXPERIMENT_PATH
EXPERIMENT_PATH


PosixPath('experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-10_16-22-14_09b78')

## Helpers


In [420]:
def read_config_from_log(experiment_path: Path):
    log_path = experiment_path / "log.txt"
    assert log_path.exists(), log_path
    with log_path.open("r") as f:
        first_line = f.readline().strip()
    return ast.literal_eval(first_line)


def get_dim_data_and_model(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    num_concepts = cfg["data"].get("num_concepts")
    num_residuals = cfg["data"].get("num_residuals")
    hid_dim = cfg["data"]["hid_dim"]
    obs_dim = cfg["data"]["obs_dim"]
    if num_concepts is None or num_residuals is None:
        raise ValueError(f"Could not find num_concepts or num_residuals in config: {cfg}")
    return num_concepts, num_residuals, hid_dim, obs_dim




def get_data_dir_name_from_log(experiment_path: Path):
    log_path = experiment_path / "log.txt"
    assert log_path.exists(), log_path

    data_dir_line = None
    with log_path.open("r") as f:
        for line in f:
            if line.startswith("data_dir:"):
                data_dir_line = line.split("data_dir:", 1)[1].strip()
                break
            if line.startswith("Loading existing synthetic dataset from"):
                data_dir_line = line.split("Loading existing synthetic dataset from", 1)[1].strip()
                break

    if data_dir_line is None:
        raise ValueError(f"Could not find data directory in {log_path}")

    return Path(data_dir_line).name


def infer_data_path(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    data_cfg = cfg["data"]
    dataset = data_cfg["dataset"]
    difficulty = data_cfg.get("experiment_type")
    data_dir_name = get_data_dir_name_from_log(experiment_path)

    candidates = []
    if difficulty is not None:
        candidates.append(Path("datasets") / dataset / difficulty / data_dir_name)
    candidates.append(Path("datasets") / dataset / data_dir_name)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError("Could not find data path. Tried: " + ", ".join(map(str, candidates)))


def load_tensor(path: Path):
    assert path.exists(), path
    return torch.load(path, map_location="cpu")


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def load_split(experiment_path: Path, data_path: Path, split: str, num_concepts=None, num_residuals=None):
    model_split = experiment_path / split
    data_split = data_path / split

    if not model_split.exists():
        raise FileNotFoundError(
            f"Missing {model_split}. Re-run inference/training after adding deterministic analysis loaders."
        )
    assert data_split.exists(), data_split
    
    
    if CONCEPT_RESIDUAL_CHANNEL:
        c_res_mu = load_tensor(model_split / "c_res_mu.pt")
        concept_residual_probs = load_tensor(model_split / "concepts_residuals_pred_probs_mean.pt")
        concept_residual_probs_std = load_tensor(model_split / "concepts_residuals_pred_probs_std.pt")
        concept_residual_sample_mean = load_tensor(model_split / "concepts_residuals_sample_mean.pt")
        
        res_mu = c_res_mu[:, num_concepts:]
        c_mu = c_res_mu[:, :num_concepts]
        concept_probs = concept_residual_probs[:, :num_concepts]
        residual_probs = concept_residual_probs[:, num_concepts:]
        concept_probs_std = concept_residual_probs_std[:, :num_concepts]
        residual_probs_std = concept_residual_probs_std[:, num_concepts:]
        concept_sample_mean = concept_residual_sample_mean[:, :num_concepts]
        residual_sample_mean = concept_residual_sample_mean[:, num_concepts:]
        
        


        out = {
            "res_mu": res_mu,
            "c_mu": c_mu,
            "concept_probs": concept_probs,
            "residual_probs": residual_probs,
            "concept_probs_std": concept_probs_std,
            "residual_probs_std": residual_probs_std,
            "concept_sample_mean": concept_sample_mean,
            "residual_sample_mean": residual_sample_mean,
            "hidden_residuals": load_tensor(data_split / "residuals.pt"),
            "hidden_residual_signal": load_tensor(data_split / "residual_signal.pt"),
            "concepts": load_tensor(data_split / "concepts.pt"),
            "y": load_tensor(data_split / "y.pt"),
            "w_hid": load_tensor(data_split / "w_hid.pt"),
            "w_obs": load_tensor(data_split / "w_obs.pt"), 
        }
    
    
        # print shapes
        print(f"res_mu shape: {out['res_mu'].shape}")
        print(f"residual_probs shape: {out['residual_probs'].shape}")
        print(f"hidden_residuals shape: {out['hidden_residuals'].shape}")
        
        
        
    
    
    
    
    
    else:
        out = {
            "res_mu": load_tensor(model_split / "res_mu.pt"),
            "residual_probs": load_tensor(model_split / "residuals_pred_probs_mean.pt"),
            "residual_probs_std": load_tensor(model_split / "residuals_pred_probs_std.pt"),
            "residual_sample_mean": load_tensor(model_split / "residuals_sample_mean.pt"),
            "hidden_residuals": load_tensor(data_split / "residuals.pt"),
            "hidden_residual_signal": load_tensor(data_split / "residual_signal.pt"),
            "concepts": load_tensor(data_split / "concepts.pt"),
            "y": load_tensor(data_split / "y.pt"),
            "w_hid": load_tensor(data_split / "w_hid.pt"),
            "w_obs": load_tensor(data_split / "w_obs.pt"),
            
        }
        
    
    
    

    n_model = out["res_mu"].shape[0]
    n_data = out["hidden_residuals"].shape[0]
    assert n_model == n_data, (
        f"{split} row mismatch: model residuals have {n_model} rows, "
        f"but hidden residual labels have {n_data}. The residual dump likely used a shuffled/drop_last loader."
    )
    assert out["residual_probs"].shape[0] == n_data
    assert out["residual_sample_mean"].shape[0] == n_data

    return out







def orientation_free_auc(y_true, score):
    y_true = to_numpy(y_true).astype(int)
    score = to_numpy(score).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan, "degenerate"
    raw_auc = roc_auc_score(y_true, score)
    if raw_auc >= 0.5:
        return raw_auc, "positive"
    return 1.0 - raw_auc, "negative"


def residual_hidden_auc_matrix(residual_scores, hidden_residuals):
    R = to_numpy(residual_scores).astype(float)
    H = to_numpy(hidden_residuals).astype(int)

    
    
    
    
    
    
    auc = np.zeros((R.shape[1], H.shape[1]), dtype=float)
    directions = np.empty((R.shape[1], H.shape[1]), dtype=object)

    for r in range(R.shape[1]):
        for h in range(H.shape[1]):
            auc[r, h], directions[r, h] = orientation_free_auc(H[:, h], R[:, r])

    return auc, directions


def plot_auc_matrix(auc, title, relevant_hidden_indices=None):
    plt.figure(figsize=(1.15 * auc.shape[1] + 3, 0.95 * auc.shape[0] + 2))
    # Plot only relevenat hidden indices if provided, otherwise plot all
    auc = auc[:, relevant_hidden_indices] if relevant_hidden_indices is not None else auc
    
    
    sns.heatmap(
        auc,
        annot=True,
        fmt=".3f",
        vmin=0.5,
        vmax=1.0,
        cmap="viridis",
        xticklabels=[f"h{j}" for j in range(auc.shape[1])],
        yticklabels=[f"r{i}" for i in range(auc.shape[0])],
    )
    plt.xlabel("True hidden residual")
    plt.ylabel("Learned residual dimension")
    plt.title(title)
    plt.tight_layout()


def match_hidden_to_residuals(residual_scores, hidden_residuals):
    """
    For each hidden residual dimension, find the best matching residual dimension based on AUC.
    """
    auc, directions = residual_hidden_auc_matrix(residual_scores, hidden_residuals)
    rows = []

    # For each hidden residual dimension, find the residual dimension with the highest AUC 
    # and record the direction.
    for h in range(auc.shape[1]):
        r = int(np.nanargmax(auc[:, h]))
        rows.append({
            "hidden_idx": h,
            "residual_idx": r,
            "auc": float(auc[r, h]),
            "direction": directions[r, h],
        })

    return pd.DataFrame(rows), auc, directions


def evaluate_fixed_matches(residual_scores, hidden_residuals, matches, threshold=0.0):
    """If validation says residual dimension r_k corresponds to hidden concept h_j, 
    does that same residual dimension recover h_j on unseen test data"""
    
    
    R = to_numpy(residual_scores).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []
    
    # row = [hidden_idx, residual_idx, direction]
    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        r = int(row.residual_idx)
        # Score is for example res_mu if that is passed in as residual_scores
        score = R[:, r]
        direction = row.direction

        if direction == "negative":
            score = -score

        auc = roc_auc_score(H[:, h], score) if len(np.unique(H[:, h])) == 2 else np.nan
        # AUC does not depend on the threshold, but accuracy and F1 do
        pred = (score >= threshold).astype(int)

        rows.append({
            "hidden_idx": h,
            "residual_idx": r,
            "direction": direction,
            "auc": float(auc),
            "accuracy_at_threshold": float(accuracy_score(H[:, h], pred)),
            "f1_at_threshold": float(f1_score(H[:, h], pred)),
        })

    return pd.DataFrame(rows)


def add_task_relevance(df, w_hid):
    """Add a column to the hidden residuals dataframe indicating whether each hidden residual is task-relevant based on w_hid."""
    w_hid_np = to_numpy(w_hid).astype(float)
    df = df.copy()
    df["w_hid"] = w_hid_np[df["hidden_idx"].values]
    df["task_relevant"] = np.abs(df["w_hid"]) > 1e-8
    df["abs_w_hid"] = np.abs(df["w_hid"])
    df["rank_abs_w"] = df["abs_w_hid"].rank(method="min", ascending=False).astype(int)
    return df



def get_task_hidden_feature_name(experiment_path: Path):
    cfg = read_config_from_log(experiment_path)
    difficulty = cfg["data"].get("experiment_type")
    if difficulty == "hard":
        return "hidden_residual_signal"
    return "hidden_residuals"


def hidden_relevance_table(split_data, experiment_path: Path):
    """
    Estimate true hidden-concept task importance from the synthetic task weights.

    The primary importance metric is abs(w_hid): the absolute ground-truth
    coefficient assigned to each hidden concept by the data-generating process.
    Signal/prevalence statistics are retained only as descriptive diagnostics.
    """
    w = to_numpy(split_data["w_hid"]).astype(float)
    H = to_numpy(split_data["hidden_residuals"]).astype(float)
    S = to_numpy(split_data["hidden_residual_signal"]).astype(float)
    y = to_numpy(split_data["y"]).astype(int)
    task_feature_name = get_task_hidden_feature_name(experiment_path)
    task_feature = H if task_feature_name == "hidden_residuals" else S

    rows = []
    for h in range(H.shape[1]):
        # These empirical terms are useful diagnostics, but they do not define
        # task importance in this notebook. Importance is ranked by abs(w_hid).
        term = w[h] * task_feature[:, h]
        signal_term = w[h] * S[:, h]
        binary_term = w[h] * H[:, h]

        if len(np.unique(H[:, h])) == 2:
            hidden_auc_y, hidden_dir_y = orientation_free_auc(y, H[:, h])
        else:
            hidden_auc_y, hidden_dir_y = np.nan, "degenerate"

        rows.append({
            "hidden_idx": h,
            "w_hid": w[h],
            "abs_w_hid": abs(w[h]),
            "prevalence": H[:, h].mean(),
            "mean_signal": S[:, h].mean(),
            "std_binary_feature": H[:, h].std(),
            "std_signal_feature": S[:, h].std(),
            "task_feature_used_for_y": task_feature_name,
            "mean_abs_binary_term": np.abs(binary_term).mean(),
            "std_binary_term": binary_term.std(),
            "mean_abs_signal_term": np.abs(signal_term).mean(),
            "std_signal_term": signal_term.std(),
            "mean_abs_task_term": np.abs(term).mean(),
            "std_task_term": term.std(),
            "hidden_auc_vs_y": hidden_auc_y,
            "hidden_direction_vs_y": hidden_dir_y,
        })

    out = pd.DataFrame(rows)
    # Rank by absolute task weight, then by std of task term, then by hidden AUC vs y
    out["rank_abs_w"] = out["abs_w_hid"].rank(method="min", ascending=False).astype(int)
    out["rank_std_task_term"] = out["std_task_term"].rank(method="min", ascending=False).astype(int)
    out["rank_hidden_auc_vs_y"] = out["hidden_auc_vs_y"].rank(method="min", ascending=False).astype(int)
    return out.sort_values("rank_abs_w")

def attach_relevance(df, relevance):
    cols = [
        "hidden_idx",
        "w_hid",
        "abs_w_hid",
        "prevalence",
        "mean_signal",
        "std_task_term",
        "mean_abs_task_term",
        "hidden_auc_vs_y",
        "rank_abs_w",
        "rank_std_task_term",
        "rank_hidden_auc_vs_y",
    ]

    # Some upstream tables may already contain older relevance columns from
    # add_task_relevance(...). Drop them before merging so pandas does not create
    # confusing suffixes like w_hid_x / w_hid_y.
    df = df.copy()
    stale_cols = [col for col in cols if col != "hidden_idx" and col in df.columns]
    if stale_cols:
        df = df.drop(columns=stale_cols)

    return df.merge(relevance[cols], on="hidden_idx", how="left")

def summarize_recovery_by_relevance(df, score_col, relevance_col="abs_w_hid", ks=(1, 3, 5, 10)):
    """
    Summarize how well the top-k most relevant hidden concepts are recovered by a given score (e.g. residual-hidden AUC or probe coefficients).
    """
    rows = []
    ordered = df.sort_values(relevance_col, ascending=False)
    for k in ks:
        top = ordered.head(k)
        rows.append({
            "top_k_by": relevance_col,
            "k": k,
            f"mean_{score_col}": top[score_col].mean(),
            f"max_{score_col}": top[score_col].max(),
            f"num_{score_col}_ge_0_7": int((top[score_col] >= 0.7).sum()),
            "hidden_indices": top["hidden_idx"].tolist(),
        })
    return pd.DataFrame(rows)



# -------------------------------------------------------------
# Test metrics of concept model and linear model
# --------------------------------------------------------------
def get_metrics_dataset_linear_model(full_model_path, dataset_difficulty, dataset="synthetic_res_scbm"):
    full_data_path = get_data_dir_name_from_log(full_model_path)
    data_dir_name = full_data_path.split("/")[-1]
    linear_models_dir = os.path.join("experiments", "linear_head", dataset, dataset_difficulty)
    for model_dir in os.listdir(linear_models_dir):
        model_name_start = data_dir_name + "_trueResUsed_False"
        if model_dir.startswith(model_name_start):
            linear_model_path = model_dir
            break
    with open(os.path.join(linear_models_dir, linear_model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"Final Test Accuracy:\s*([0-9.]+)", test_line)[0]
        
    return float(y_accuracy)


def test_metrics(full_model_path):
    metrics = {}
    with open(os.path.join(full_model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"y_accuracy:\s*([0-9.]+)", test_line)[0]
        c_accuracy = re.findall(r"c_accuracy:\s*([0-9.]+)", test_line)[0]
        c_auc = re.findall(r"c_AUROC:\s*([0-9.]+)", test_line)[0]
        metrics["y_accuracy"] = float(y_accuracy)
        metrics["c_accuracy"] = float(c_accuracy)
        metrics["c_auc"] = float(c_auc)
    return metrics  










# Model performance

In [423]:
metrics = {}

concept_model_metrics = test_metrics(EXPERIMENT_PATH)

linear_model_y_accuracy = get_metrics_dataset_linear_model(EXPERIMENT_PATH, dataset_difficulty="hard")

for key, value in concept_model_metrics.items():
    metrics[key] = value

metrics["linear_model_y_accuracy"] = linear_model_y_accuracy

print(EXPERIMENT_PATH)
print(get_data_dir_name_from_log(EXPERIMENT_PATH))
num_concepts, num_residuals, hid_dim, obs_dim = get_dim_data_and_model(EXPERIMENT_PATH)
print(f"Number of concepts: {num_concepts}, number of residuals: {num_residuals}, obs_dim: {obs_dim}, hid_dim: {hid_dim}")


num_concepts, num_residuals, hid_dim, obs_dim
df_test_metrics = pd.DataFrame([metrics])
df_test_metrics.head()

DATA_PATH = infer_data_path(EXPERIMENT_PATH)


train_split = load_split(EXPERIMENT_PATH, DATA_PATH, split="train")

w_obs = train_split["w_obs"]
w_hid = train_split["w_hid"]


df_weight_dict = {"dimension": [], "weight": [], "absolute_weight": []}
for c, w in enumerate(w_obs):
    df_weight_dict["dimension"].append(f"c{c}")
    df_weight_dict["weight"].append(w.item())
    df_weight_dict["absolute_weight"].append(abs(w.item()))

for h, w in enumerate(w_hid):
    df_weight_dict["dimension"].append(f"h{h}")
    df_weight_dict["weight"].append(w.item())
    df_weight_dict["absolute_weight"].append(abs(w.item()))

print("Data generating weights:")
df_weights = pd.DataFrame.from_dict(df_weight_dict)
df_weights.sort_values("absolute_weight", ascending=False, inplace=True)
# Drop rows with zero weight for better visualization
df_weights = df_weights[df_weights["absolute_weight"] > 1e-8]
df_weights.head(40)

print(f"Task relevant hidden residuals (non-zero w_hid): {[f'h{h}' for h in range(len(w_hid)) if abs(w_hid[h].item()) > 1e-8]}")
print(f"Task relevant observed concepts (non-zero w_obs): {[f'c{c}' for c in range(len(w_obs)) if abs(w_obs[c].item()) > 1e-8]}")


path = Path("/Users/stephenmarcou/Documents/ETH Zurich/Cambridge/Code/SCBM_implementation/experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-10_16-22-14_09b78/test/c_res_mu.pt")
c_res_mu = load_tensor(path)

print(c_res_mu.shape)

experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-10_16-22-14_09b78
cluster_a_0.5_b_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_r_sparsity_0.25_c_sparsity_0.3_sigmax_0.5_seed_0
Number of concepts: 10, number of residuals: 20, obs_dim: 10, hid_dim: 20
res_mu shape: torch.Size([30000, 20])
residual_probs shape: torch.Size([30000, 30])
hidden_residuals shape: torch.Size([30000, 20])
Data generating weights:
Task relevant hidden residuals (non-zero w_hid): ['h2', 'h7', 'h9', 'h18', 'h19']
Task relevant observed concepts (non-zero w_obs): ['c0', 'c3', 'c7']
torch.Size([10000, 20])


## Load Split-Aligned Data

The shape assertions here are intentional. If they fail for train, the run was probably produced before the deterministic analysis loaders were added.


In [391]:
DATA_PATH = infer_data_path(EXPERIMENT_PATH)
print("Experiment:", EXPERIMENT_PATH)
print("Data path:", DATA_PATH)


num_concepts, num_residuals, _,_ = get_dim_data_and_model(EXPERIMENT_PATH)

if CONCEPT_RESIDUAL_CHANNEL:
    splits = {split: load_split(EXPERIMENT_PATH, DATA_PATH, split, num_concepts=num_concepts, num_residuals=num_residuals) 
              for split in ["train", "val", "test"]}

else:
    splits = {split: load_split(EXPERIMENT_PATH, DATA_PATH, split) for split in ["train", "val", "test"]}

for split, data in splits.items():
    print(f"{split}")
    for key in ["res_mu", "residual_probs", "residual_sample_mean", "hidden_residuals", "hidden_residual_signal", "concepts", "y"]:
        print(f"  {key:24s} {tuple(data[key].shape)}")


relevant_hidden = torch.where(w_hid.abs() > 1e-8)[0].tolist()


print("Task-relevant hidden residual indices:", relevant_hidden)


Experiment: experiments/scbm_residual/synthetic_res_scbm/hard/alpha_0.5_beta_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_hard_hid20_R20_hidden_strong_rrcorr05_200_epochs_save_c_res_2026-06-10_16-22-14_09b78
Data path: datasets/synthetic_res_scbm/hard/cluster_a_0.5_b_2.0_rho_cr0.0_rho_cc0.0_rho_rr0.5_r_sparsity_0.25_c_sparsity_0.3_sigmax_0.5_seed_0
train
  res_mu                   (30000, 10)
  residual_probs           (30000, 20)
  residual_sample_mean     (30000, 20)
  hidden_residuals         (30000, 20)
  hidden_residual_signal   (30000, 20)
  concepts                 (30000, 10)
  y                        (30000,)
val
  res_mu                   (10000, 10)
  residual_probs           (10000, 20)
  residual_sample_mean     (10000, 20)
  hidden_residuals         (10000, 20)
  hidden_residual_signal   (10000, 20)
  concepts                 (10000, 10)
  y                        (10000,)
test
  res_mu                   (10000, 10)
  residual_probs           (10000, 20)
  residual_sample_mean     

## Direct Residual Baseline: AUC Matrices

Use `res_mu` as the main representation. `residual_probs` is included as a secondary check.


In [373]:
auc_tables = {}

for rep_name in ["res_mu", "residual_probs"]:
    print("=" * 80)
    print(rep_name)
    auc_tables[rep_name] = {}

    # Plot AUC matrix for trian, val, test splits, where AUC computed between true hidden concepts and res_mu or residual_probs
    for split in ["train", "val", "test"]:
        auc, directions = residual_hidden_auc_matrix(splits[split][rep_name], splits[split]["hidden_residuals"])
        auc_tables[rep_name][split] = auc

        best = np.nanmax(auc, axis=0)
        print(f"{split} best AUC per hidden residual:", np.round(best, 4))
        print(f"{split} mean best AUC:", round(float(np.nanmean(best)), 4))
        if relevant_hidden:
            print(f"{split} mean best AUC, task-relevant only:", round(float(np.nanmean(best[relevant_hidden])), 4))

        #plot_auc_matrix(auc, f"{split}: {rep_name} vs true hidden residuals", relevant_hidden_indices=relevant_hidden)
        #plt.show()


res_mu
train best AUC per hidden residual: [0.5392 0.5105 0.746  0.5177 0.6193 0.6404 0.6213 0.6105 0.5335 0.66
 0.5214 0.5298 0.5857 0.6192 0.5263 0.527  0.5621 0.562  0.7076 0.55  ]
train mean best AUC: 0.5845
train mean best AUC, task-relevant only: 0.6548
val best AUC per hidden residual: [0.5526 0.511  0.7406 0.5174 0.6168 0.6409 0.6104 0.6067 0.5388 0.6516
 0.5286 0.5365 0.5941 0.6183 0.521  0.5191 0.5602 0.5609 0.6947 0.5462]
val mean best AUC: 0.5833
val mean best AUC, task-relevant only: 0.648
test best AUC per hidden residual: [0.5409 0.5111 0.732  0.5189 0.6282 0.6471 0.6254 0.6018 0.5407 0.6716
 0.5281 0.518  0.5912 0.6274 0.5342 0.5229 0.5734 0.5701 0.7005 0.5455]
test mean best AUC: 0.5864
test mean best AUC, task-relevant only: 0.6503
residual_probs
train best AUC per hidden residual: [0.5393 0.5163 0.7461 0.5177 0.6196 0.6406 0.6213 0.6106 0.5339 0.6603
 0.5214 0.53   0.5858 0.6188 0.5264 0.5273 0.5643 0.5668 0.7074 0.5505]
train mean best AUC: 0.5852
train mean best AU

## Choose Matches On Validation, Evaluate On Test

This is the leakage-safe direct baseline. Validation chooses which learned residual dimension corresponds to each hidden residual. Test reports those fixed choices.

Use RES_MU


In [374]:
matches_train, auc_train, directions_train = match_hidden_to_residuals(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
)

matches_val, auc_val, directions_val = match_hidden_to_residuals(
    splits["val"]["res_mu"],
    splits["val"]["hidden_residuals"],
)

match_comparison = matches_val.merge(
    matches_train,
    on="hidden_idx",
    suffixes=("_val", "_train"),
)
match_comparison["same_residual_match"] = match_comparison["residual_idx_val"] == match_comparison["residual_idx_train"]
match_comparison = add_task_relevance(match_comparison.rename(columns={"hidden_idx": "hidden_idx"}), w_hid)


# Sort by abs(w_hid) to prioritize task-relevant hidden residuals
match_comparison = add_task_relevance(match_comparison, splits["test"]["w_hid"])
match_comparison = match_comparison.assign(abs_w_hid=match_comparison["w_hid"].abs())
match_comparison = match_comparison.sort_values("abs_w_hid", ascending=False)

display(match_comparison)



,hidden_idx,residual_idx_val,auc_val,direction_val,residual_idx_train,auc_train,direction_train,same_residual_match,w_hid,task_relevant,abs_w_hid,rank_abs_w
18,18,3,0.694672,negative,3,0.707593,negative,True,-0.705692,True,0.705692,1
2,2,6,0.740553,positive,6,0.745957,positive,True,-0.648612,True,0.648612,2
9,9,1,0.651557,positive,1,0.659975,positive,True,0.230210,True,0.230210,3
19,19,0,0.546242,positive,0,0.549955,positive,True,-0.150507,True,0.150507,4
7,7,3,0.606747,negative,3,0.610544,negative,True,0.075182,True,0.075182,5
12,12,7,0.594145,negative,7,0.585707,negative,True,0.000000,False,0.000000,6
17,17,3,0.560857,negative,6,0.561980,positive,False,0.000000,False,0.000000,6
16,16,7,0.560201,negative,9,0.562063,positive,False,0.000000,False,0.000000,6
15,15,9,0.519063,negative,3,0.526952,positive,False,0.000000,False,0.000000,6
14,14,7,0.521027,positive,1,0.526256,positive,False,0.000000,False,0.000000,6


In [375]:
# Evaluate the validation-chosen raw residual/hidden matches on the held-out test split.
test_eval = evaluate_fixed_matches(
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
    matches_val,
    threshold=0.0,
)

# Build one canonical hidden-concept relevance table for the rest of the notebook.
# Primary task relevance is abs(w_hid); the extra columns are descriptive diagnostics.
relevance = hidden_relevance_table(splits["test"], EXPERIMENT_PATH)

# Canonical raw-axis recovery table used by downstream comparison cells.
# Rename auc -> axis_auc so the column name is explicit when compared to probes/SAE.
recovery_eval = (
    attach_relevance(test_eval, relevance)
    .rename(columns={"auc": "axis_auc"})
    .sort_values("rank_abs_w")
)
recovery_eval["task_relevant"] = recovery_eval["abs_w_hid"] > 1e-8

print("Test evaluation of validation-chosen residual-hidden matches")
display(recovery_eval[[
    "hidden_idx",
    "residual_idx",
    "direction",
    "axis_auc",
    "accuracy_at_threshold",
    "f1_at_threshold",
    "w_hid",
    "abs_w_hid",
    "task_relevant",
]])

summary = pd.Series({
    "mean_auc_all_hidden": recovery_eval["axis_auc"].mean(),
    "mean_auc_task_relevant_hidden": recovery_eval.loc[recovery_eval["abs_w_hid"] > 1e-8, "axis_auc"].mean(),
    "num_hidden_auc_ge_0_7": int((recovery_eval["axis_auc"] >= 0.7).sum()),
    "num_task_relevant_hidden_auc_ge_0_7": int(((recovery_eval["axis_auc"] >= 0.7) & (recovery_eval["abs_w_hid"] > 1e-8)).sum()),
    "mean_accuracy_at_zero": recovery_eval["accuracy_at_threshold"].mean(),
})
summary


Test evaluation of validation-chosen residual-hidden matches


,hidden_idx,residual_idx,direction,axis_auc,accuracy_at_threshold,f1_at_threshold,w_hid,abs_w_hid,task_relevant
18,18,3,negative,0.700518,0.6520,0.643150,-0.705692,0.705692,True
2,2,6,positive,0.732040,0.6713,0.666937,-0.648612,0.648612,True
9,9,1,positive,0.671554,0.6185,0.629935,0.230210,0.230210,True
19,19,0,positive,0.544257,0.5300,0.514162,-0.150507,0.150507,True
7,7,3,negative,0.601763,0.5714,0.559507,0.075182,0.075182,True
4,4,1,negative,0.628176,0.5925,0.581321,0.000000,0.000000,False
5,5,7,negative,0.647120,0.6039,0.595858,0.000000,0.000000,False
6,6,3,positive,0.625382,0.5927,0.602518,0.000000,0.000000,False
8,8,2,positive,0.540746,0.5319,0.545578,0.000000,0.000000,False
1,1,6,positive,0.498767,0.4979,0.492161,0.000000,0.000000,False


mean_auc_all_hidden                    0.585560
mean_auc_task_relevant_hidden          0.650027
num_hidden_auc_ge_0_7                  2.000000
num_task_relevant_hidden_auc_ge_0_7    2.000000
mean_accuracy_at_zero                  0.562025
dtype: float64

## Confusion matrix

In [414]:
def find_matches(residual_scores, hidden_residuals, abs_w_hid, top_k=1):
    """
    For each hidden residual dimension, find the top-k matching residual dimensions based on AUC.
    """
    auc, directions = residual_hidden_auc_matrix(residual_scores, hidden_residuals)
    matches = []

    for h in range(auc.shape[1]):
        top_r_indices = np.argsort(auc[:, h])[-top_k:][::-1]  # Get indices of top-k AUC scores
        #print(len(top_r_indices))
        for r in top_r_indices:
            matches.append({
                "hidden_idx": h,
                "residual_idx": r,
                "auc": float(auc[r, h]),
                "direction": directions[r, h],
                "abs_w_hid": abs_w_hid[h].item(),
            })
    df = pd.DataFrame(matches)
    df.sort_values("abs_w_hid", ascending=False, inplace=True)
    return df


test_true_hidden_residuals = splits["test"]["hidden_residuals"]
test_residual_probs = splits["test"]["res_mu"]
test_w_hid = splits["test"]["w_hid"]
test_abs_w_hid = test_w_hid.abs()
df_auc = find_matches(test_residual_probs, test_true_hidden_residuals, test_abs_w_hid, top_k=100)

print(test_residual_probs.shape, test_true_hidden_residuals.shape)

df_auc.head(40)

# top 2 hidden hidden concepts
#top_hidden_idx = list(df_auc.head(2)["hidden_idx"])


torch.Size([10000, 10]) torch.Size([10000, 20])


,hidden_idx,residual_idx,auc,direction,abs_w_hid
186,18,6,0.690681,positive,0.705692
181,18,9,0.699438,positive,0.705692
185,18,8,0.690876,negative,0.705692
182,18,7,0.694508,negative,0.705692
187,18,0,0.689856,positive,0.705692
188,18,2,0.689426,negative,0.705692
189,18,5,0.687164,positive,0.705692
180,18,3,0.700518,negative,0.705692
184,18,1,0.693903,negative,0.705692
183,18,4,0.693927,negative,0.705692


In [408]:
auc, directions = residual_hidden_auc_matrix(test_residual_probs, test_true_hidden_residuals)

print(auc)

[[0.53726184 0.50244879 0.72901317 0.51228001 0.62076666 0.63953566
  0.61256035 0.59294127 0.53309484 0.65724021 0.51708051 0.51696216
  0.58132222 0.62084578 0.52721042 0.51482442 0.56840841 0.56574796
  0.68985565 0.54425721]
 [0.53951711 0.50561572 0.72645617 0.51629325 0.62817571 0.64465415
  0.62063551 0.5989541  0.53931629 0.67155442 0.52217267 0.51796248
  0.59116566 0.62400057 0.53420981 0.51426566 0.57215553 0.5629384
  0.69390263 0.54229038]
 [0.53256093 0.50200598 0.72710808 0.51710921 0.6226732  0.64660988
  0.61979862 0.59389016 0.54074606 0.66149717 0.51780046 0.51758566
  0.58753872 0.61874459 0.52803381 0.509463   0.56883394 0.56494747
  0.68942621 0.5450081 ]
 [0.54028776 0.50969277 0.71999699 0.51369303 0.62517447 0.64026877
  0.62538191 0.60176324 0.53152352 0.66943721 0.52108996 0.5062952
  0.58947888 0.62735731 0.53237042 0.52149124 0.56888884 0.57007853
  0.70051847 0.53999555]
 [0.53905564 0.50232685 0.71772917 0.51296488 0.6228283  0.64501315
  0.61923952 0.598

## Direct Binary Interpretation

This treats each learned residual dimension as a binary discovered concept. This section is diagnostic: it reports the best test-set axis for each hidden residual, so do not use it to choose final matches. The leakage-safe report is the validation-chosen table above.


In [376]:
def binary_direct_recovery(residual_scores, hidden_residuals, score_type="mu"):
    R = to_numpy(residual_scores).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    threshold = 0.0 if score_type == "mu" else 0.5

    rows = []
    for h in range(H.shape[1]):
        for r in range(R.shape[1]):
            score = R[:, r]
            raw_auc = roc_auc_score(H[:, h], score)

            pred_pos = (score >= threshold).astype(int)
            pred_neg = (score < threshold).astype(int)
            acc_pos = accuracy_score(H[:, h], pred_pos)
            acc_neg = accuracy_score(H[:, h], pred_neg)

            if acc_pos >= acc_neg:
                best_acc = acc_pos
                best_f1 = f1_score(H[:, h], pred_pos)
                direction = "positive"
            else:
                best_acc = acc_neg
                best_f1 = f1_score(H[:, h], pred_neg)
                direction = "negative"

            rows.append({
                "hidden_idx": h,
                "residual_idx": r,
                "orientation_free_auc": max(raw_auc, 1 - raw_auc),
                "best_threshold_accuracy": best_acc,
                "best_threshold_f1": best_f1,
                "direction": direction,
            })

    return pd.DataFrame(rows)

for rep_name, score_type in [("res_mu", "mu"), ("residual_probs", "prob")]:
    print("=" * 80)
    print(rep_name)
    df_bin = binary_direct_recovery(splits["test"][rep_name], splits["test"]["hidden_residuals"], score_type=score_type)
    best = df_bin.sort_values("orientation_free_auc", ascending=False).groupby("hidden_idx").head(1).reset_index(drop=True)
    best = add_task_relevance(best, splits["test"]["w_hid"])
    #display(best)


res_mu


,hidden_idx,residual_idx,orientation_free_auc,best_threshold_accuracy,best_threshold_f1,direction,w_hid,task_relevant,abs_w_hid,rank_abs_w
0,2,6,0.732040,0.6713,0.666937,positive,-0.648612,True,0.648612,2
1,18,3,0.700518,0.6520,0.643150,negative,-0.705692,True,0.705692,1
2,9,1,0.671554,0.6185,0.629935,positive,0.230210,True,0.230210,3
3,5,7,0.647120,0.6039,0.595858,negative,0.000000,False,0.000000,6
4,4,1,0.628176,0.5925,0.581321,negative,0.000000,False,0.000000,6
5,13,3,0.627357,0.5931,0.581249,negative,0.000000,False,0.000000,6
6,6,3,0.625382,0.5927,0.602518,positive,0.000000,False,0.000000,6
7,7,3,0.601763,0.5714,0.559507,negative,0.075182,True,0.075182,5
8,12,1,0.591166,0.5637,0.546041,negative,0.000000,False,0.000000,6
9,16,9,0.573417,0.5520,0.542110,positive,0.000000,False,0.000000,6


residual_probs


,hidden_idx,residual_idx,orientation_free_auc,best_threshold_accuracy,best_threshold_f1,direction,w_hid,task_relevant,abs_w_hid,rank_abs_w
0,2,6,0.732524,0.6712,0.665650,negative,-0.648612,True,0.648612,2
1,18,13,0.700490,0.6530,0.643810,negative,-0.705692,True,0.705692,1
2,9,11,0.671565,0.6202,0.631334,positive,0.230210,True,0.230210,3
3,5,5,0.649800,0.6054,0.592860,positive,0.000000,False,0.000000,6
4,4,8,0.629495,0.5960,0.586150,negative,0.000000,False,0.000000,6
5,13,13,0.627475,0.5925,0.580200,negative,0.000000,False,0.000000,6
6,6,13,0.626076,0.5925,0.602710,positive,0.000000,False,0.000000,6
7,7,13,0.602338,0.5724,0.560082,negative,0.075182,True,0.075182,5
8,12,11,0.591567,0.5648,0.547515,negative,0.000000,False,0.000000,6
9,16,1,0.576173,0.5526,0.543935,negative,0.000000,False,0.000000,6


## Distributed Probe Baseline

This asks whether hidden residual information is present in the residual vector even when it is not axis-aligned. The probe is trained on train residuals and evaluated on test residuals.


In [377]:
def distributed_probe_train_test(train_scores, train_hidden, test_scores, test_hidden):
    X_train = to_numpy(train_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    # For each hidden concept train a logistic regression on residual channel (res_mu)
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
        )
        clf.fit(X_train, H_train[:, h])

        prob = clf.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "hidden_idx": h,
            "distributed_auc": roc_auc_score(H_test[:, h], prob),
            "distributed_accuracy": accuracy_score(H_test[:, h], pred),
            "distributed_f1": f1_score(H_test[:, h], pred),
        })

    return pd.DataFrame(rows)

probe_eval = distributed_probe_train_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)






probe_recovery_eval = add_task_relevance(probe_eval, splits["test"]["w_hid"])
#probe_recovery_eval = attach_relevance(probe_eval, relevance)

print("Distributed-probe recovery among top-k hidden concepts by abs(w_hid)")
# This gives the mean auc for top k hidden residuals ranked by abs(w_hid)
display(summarize_recovery_by_relevance(probe_recovery_eval, "distributed_auc", "abs_w_hid"))



probe_recovery_eval.sort_values("rank_abs_w")


Distributed-probe recovery among top-k hidden concepts by abs(w_hid)


,top_k_by,k,mean_distributed_auc,max_distributed_auc,num_distributed_auc_ge_0_7,hidden_indices
0,abs_w_hid,1,0.713136,0.713136,1,[18]
1,abs_w_hid,3,0.714884,0.746453,2,"[18, 2, 9]"
2,abs_w_hid,5,0.666833,0.746453,2,"[18, 2, 9, 19, 7]"
3,abs_w_hid,10,0.629327,0.746453,2,"[18, 2, 9, 19, 7, 12, 17, 16, 15, 14]"


,hidden_idx,distributed_auc,distributed_accuracy,distributed_f1,w_hid,task_relevant,abs_w_hid,rank_abs_w
18,18,0.713136,0.6594,0.657963,-0.705692,True,0.705692,1
2,2,0.746453,0.6806,0.673615,-0.648612,True,0.648612,2
9,9,0.685063,0.6344,0.641639,0.230210,True,0.230210,3
19,19,0.571436,0.5487,0.541688,-0.150507,True,0.150507,4
7,7,0.618079,0.5900,0.587442,0.075182,True,0.075182,5
4,4,0.631822,0.5964,0.592323,0.000000,False,0.000000,6
5,5,0.651168,0.6033,0.597545,0.000000,False,0.000000,6
6,6,0.644138,0.6048,0.607235,0.000000,False,0.000000,6
8,8,0.604129,0.5745,0.579753,0.000000,False,0.000000,6
1,1,0.564609,0.5466,0.556794,0.000000,False,0.000000,6


## Strict Top-k Residual Probe Recovery

This supervised diagnostic asks whether each hidden concept is recoverable from only the top-k residual dimensions selected by a dense probe. It separates axis-like, sparse-mixed, and broadly distributed residual encodings.


In [378]:
# -----------------------------
# Strict top-k residual probe recovery
# -----------------------------
# Dense probes show whether hidden concept information is linearly present in the
# residual channel. This section asks a stricter question: how many residual
# dimensions are needed to recover each hidden concept?
#
# Protocol for each hidden concept h:
# 1. Fit a dense logistic probe h ~ res_mu on train.
# 2. Rank residual dimensions by absolute dense-probe coefficient.
# 3. Refit probes using only the top-k ranked dimensions.
# 4. Evaluate every fixed top-k subset on test.
#
# Hidden labels are used here, so this is a recoverability diagnostic rather
# than an unsupervised discovery method.

TOPK_PROBE_KS = [1, 2, 3, 5, 10, 20, 50]


def fit_topk_probe_for_hidden(X_train, y_train, X_test, y_test, ks=TOPK_PROBE_KS):
    """Rank residual dimensions with a dense probe, then test strict top-k refits.
    Only fit with the top-k dimensions with highest absolute coefficient in the dense probe.
    """
    n_dims = X_train.shape[1]
    ks = sorted({min(k, n_dims) for k in ks})

    dense_clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
    )
    dense_clf.fit(X_train, y_train)
    dense_prob = dense_clf.predict_proba(X_test)[:, 1]
    dense_auc = roc_auc_score(y_test, dense_prob)

    coef = dense_clf.named_steps["logisticregression"].coef_[0]
    
    ranked_dims = np.argsort(-np.abs(coef)).astype(int).tolist()

    rows = []
    for k in ks:
        # Select the top-k dimensions based on the dense probe's absolute coefficient ranking.

        selected_dims = ranked_dims[:k]
        topk_clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
        )
        topk_clf.fit(X_train[:, selected_dims], y_train)

        prob = topk_clf.predict_proba(X_test[:, selected_dims])[:, 1]
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "k": k,
            "topk_auc": roc_auc_score(y_test, prob),
            "topk_accuracy": accuracy_score(y_test, pred),
            "topk_f1": f1_score(y_test, pred, zero_division=0),
            "dense_auc_from_same_probe": dense_auc,
            "topk_gap_to_dense": dense_auc - roc_auc_score(y_test, prob),
            "selected_dims": selected_dims,
        })

    return rows, ranked_dims, coef


def strict_topk_probe_train_test(train_scores, train_hidden, test_scores, test_hidden, ks=TOPK_PROBE_KS):
    X_train = to_numpy(train_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    coef_rows = []
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        topk_rows, ranked_dims, coef = fit_topk_probe_for_hidden(
            X_train,
            H_train[:, h],
            X_test,
            H_test[:, h],
            ks=ks,
        )
        for row in topk_rows:
            rows.append({"hidden_idx": h, **row})

        coef_rows.append({
            "hidden_idx": h,
            "ranked_dims_by_dense_coef": ranked_dims,
            "top10_dense_coef_dims": ranked_dims[:10],
            "dense_coef_l1_norm": float(np.abs(coef).sum()),
            "dense_coef_l2_norm": float(np.sqrt((coef ** 2).sum())),
        })

    return pd.DataFrame(rows), pd.DataFrame(coef_rows)


topk_probe_long, topk_probe_coef_info = strict_topk_probe_train_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)

topk_probe_long = attach_relevance(topk_probe_long, relevance)

# Wide table: one row per hidden concept, one AUC column per k.
topk_auc_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_auc")
    .rename(columns=lambda k: f"top{k}_auc")
    .reset_index()
)

topk_gap_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_gap_to_dense")
    .rename(columns=lambda k: f"top{k}_gap_to_dense")
    .reset_index()
)

topk_probe_recovery_eval = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(topk_auc_wide, on="hidden_idx", how="left")
    .merge(topk_gap_wide, on="hidden_idx", how="left")
    .merge(topk_probe_coef_info, on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

candidate_display_cols = [
    "hidden_idx",
    "abs_w_hid",
    "rank_abs_w",
    "axis_auc",
    "top1_auc",
    "top2_auc",
    "top3_auc",
    "top5_auc",
    "top10_auc",
    "top20_auc",
    "top50_auc",
    "distributed_auc",
    "top10_dense_coef_dims",
]
display_cols = [col for col in candidate_display_cols if col in topk_probe_recovery_eval.columns]

print("Strict top-k residual probe recovery ranked by abs(w_hid)")
display(topk_probe_recovery_eval[display_cols])

# Summarize top-k hidden concepts by true task weight, for each strict residual subset size.
summary_rows = []
for hidden_k in [1, 3, 5, 10]:
    top_hidden = topk_probe_recovery_eval.head(hidden_k)
    row = {"top_hidden_by_abs_w": hidden_k}
    row["raw_axis"] = top_hidden["axis_auc"].mean()
    for residual_k in TOPK_PROBE_KS:
        col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
        row[f"top{residual_k}_residual_dims"] = top_hidden[col].mean()
    row["dense_probe"] = top_hidden["distributed_auc"].mean()
    summary_rows.append(row)

topk_probe_summary = pd.DataFrame(summary_rows)
print("Mean hidden recovery as residual subset size increases")
display(topk_probe_summary)

# Estimate how many residual dimensions are needed to get close to the dense probe.
# A hidden concept is considered close once top-k AUC is within 0.01/0.03/0.05 of dense.
close_rows = []
for _, row in topk_probe_recovery_eval.iterrows():
    out = {
        "hidden_idx": int(row["hidden_idx"]),
        "abs_w_hid": row["abs_w_hid"],
        "rank_abs_w": int(row["rank_abs_w"]),
        "axis_auc": row["axis_auc"],
        "dense_probe_auc": row["distributed_auc"],
    }
    for tol in [0.01, 0.03, 0.05]:
        needed = np.nan
        for residual_k in TOPK_PROBE_KS:
            col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
            if row[col] >= row["distributed_auc"] - tol:
                needed = residual_k
                break
        out[f"dims_needed_within_{tol:.2f}_auc"] = needed
    close_rows.append(out)

topk_compactness = pd.DataFrame(close_rows).sort_values("rank_abs_w")
print("Residual dimensions needed to approach dense-probe AUC")
display(topk_compactness)


Strict top-k residual probe recovery ranked by abs(w_hid)


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,top1_auc,top2_auc,top3_auc,top5_auc,top10_auc,distributed_auc,top10_dense_coef_dims
0,18,0.705692,1,0.700518,0.699438,0.701194,0.707575,0.712213,0.713136,0.713136,"[9, 3, 6, 8, 0, 7, 5, 1, 4, 2]"
1,2,0.648612,2,0.732040,0.732040,0.736530,0.736621,0.743528,0.746453,0.746453,"[6, 3, 9, 8, 4, 0, 7, 2, 1, 5]"
2,9,0.230210,3,0.671554,0.669437,0.671254,0.674431,0.680877,0.685063,0.685063,"[3, 1, 8, 0, 7, 6, 5, 2, 9, 4]"
3,19,0.150507,4,0.544257,0.536699,0.564048,0.566974,0.570042,0.571436,0.571436,"[9, 6, 0, 8, 3, 2, 7, 4, 5, 1]"
4,7,0.075182,5,0.601763,0.601763,0.613387,0.613397,0.615819,0.618079,0.618079,"[3, 7, 0, 8, 1, 6, 5, 4, 2, 9]"
17,17,0.000000,6,0.570079,0.569296,0.569789,0.577030,0.581268,0.583489,0.583489,"[6, 3, 1, 7, 0, 5, 8, 4, 9, 2]"
16,16,0.000000,6,0.572152,0.573417,0.573431,0.580624,0.584914,0.589729,0.589729,"[9, 7, 8, 6, 2, 5, 4, 1, 0, 3]"
15,15,0.000000,6,0.522905,0.522905,0.521621,0.571176,0.579086,0.594516,0.594516,"[9, 3, 6, 4, 8, 2, 1, 0, 7, 5]"
14,14,0.000000,6,0.533031,0.523818,0.556815,0.560665,0.568035,0.573579,0.573579,"[6, 1, 3, 9, 5, 7, 4, 8, 2, 0]"
13,13,0.000000,6,0.627357,0.612771,0.641634,0.643866,0.644682,0.647779,0.647779,"[8, 3, 9, 2, 5, 6, 4, 1, 7, 0]"


Mean hidden recovery as residual subset size increases


,top_hidden_by_abs_w,raw_axis,top1_residual_dims,top2_residual_dims,top3_residual_dims,top5_residual_dims,top10_residual_dims,top20_residual_dims,top50_residual_dims,dense_probe
0,1,0.700518,0.699438,0.701194,0.707575,0.712213,0.713136,0.713136,0.713136,0.713136
1,3,0.701371,0.700305,0.702993,0.706209,0.712206,0.714884,0.714884,0.714884,0.714884
2,5,0.650027,0.647875,0.657283,0.659800,0.664496,0.666833,0.666833,0.666833,0.666833
3,10,0.607566,0.604158,0.614970,0.623236,0.628046,0.632326,0.632326,0.632326,0.632326


Residual dimensions needed to approach dense-probe AUC


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,dense_probe_auc,dims_needed_within_0.01_auc,dims_needed_within_0.03_auc,dims_needed_within_0.05_auc
0,18,0.705692,1,0.700518,0.713136,3,1,1
1,2,0.648612,2,0.732040,0.746453,2,1,1
2,9,0.230210,3,0.671554,0.685063,5,1,1
3,19,0.150507,4,0.544257,0.571436,2,2,1
4,7,0.075182,5,0.601763,0.618079,2,1,1
17,4,0.000000,6,0.628176,0.631822,2,1,1
16,5,0.000000,6,0.647120,0.651168,1,1,1
15,6,0.000000,6,0.625382,0.644138,3,1,1
14,8,0.000000,6,0.540746,0.604129,5,3,2
13,3,0.000000,6,0.518865,0.539993,2,1,1


## Task-Guided Residual Direction Discovery

This section turns the recoverability diagnostic into a discovery protocol. Candidate residual directions are learned without hidden labels, then a sparse task head selects directions using only `y`. Hidden labels are used only afterward to evaluate whether the selected directions recover the most task-relevant hidden concepts.


In [379]:
# -----------------------------
# Task-guided residual direction discovery
# -----------------------------
# Goal: discover task-relevant residual directions without using hidden labels.
#
# The protocol is intentionally split into three stages:
# 1. Candidate generation from res_mu only: raw axes, PCA, SparsePCA, and a combined pool.
# 2. Task selection using only y: train a sparse logistic head from candidate scores to y.
# 3. Hidden-concept evaluation: after discovery, match selected directions to hidden concepts
#    on validation and report fixed-match recovery on test.
#
# This makes the method a genuine task-guided discovery method: hidden labels are not used
# to create, rank, or select candidate directions.

TASK_DISCOVERY_N_COMPONENTS = min(50, to_numpy(splits["train"]["res_mu"]).shape[1])
TASK_DISCOVERY_TOP_MS = [5, 10, 20, 50]
TASK_HEAD_C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
TASK_HEAD_AUC_TOL = 0.005
TASK_DIRECTION_NORM_EPS = 1e-8


def standardize_residual_splits_for_discovery(train_scores, val_scores, test_scores):
    """Standardize residual vectors using train statistics only."""
    X_train = to_numpy(train_scores).astype(float)
    X_val = to_numpy(val_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)

    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std, mean, std


def normalize_direction_rows(directions):
    """Normalize concept/direction vectors so scores are comparable across generators."""
    directions = np.asarray(directions, dtype=float)
    norms = np.linalg.norm(directions, axis=1, keepdims=True) + TASK_DIRECTION_NORM_EPS
    return directions / norms


def build_residual_direction_candidates(X_train_std, n_components=TASK_DISCOVERY_N_COMPONENTS):
    """Create candidate residual directions without using y or hidden labels."""
    n_dims = X_train_std.shape[1]
    n_components = min(n_components, n_dims)

    # Raw axes: each original residual coordinate is a candidate concept direction.
    raw_dirs = np.eye(n_dims)

    # PCA directions: orthogonal directions that explain variance in the residual channel.
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_train_std)
    pca_dirs = pca.components_

    # SparsePCA directions: variance-seeking directions with sparse loadings.
    # These are a direct test of whether residual concepts are sparse rotated axes.
    sparse_pca = SparsePCA(
        n_components=n_components,
        alpha=1.0,
        ridge_alpha=0.01,
        max_iter=1000,
        tol=1e-4,
        random_state=0,
        n_jobs=1,
    )
    sparse_pca.fit(X_train_std)
    sparse_pca_dirs = sparse_pca.components_

    candidate_sets = {
        "raw_axes": normalize_direction_rows(raw_dirs),
        "pca": normalize_direction_rows(pca_dirs),
        "sparse_pca": normalize_direction_rows(sparse_pca_dirs),
    }

    # Combined pool lets the task head choose among all simple candidate generators.
    candidate_sets["combined_raw_pca_sparsepca"] = normalize_direction_rows(
        np.vstack([candidate_sets["raw_axes"], candidate_sets["pca"], candidate_sets["sparse_pca"]])
    )
    return candidate_sets


def project_residual_directions(X_std, directions):
    """Convert residual vectors into discovered concept scores."""
    return X_std @ directions.T


def fit_sparse_task_head(Z_train, y_train, Z_val, y_val, c_grid=TASK_HEAD_C_GRID):
    """
    Train sparse task heads and choose a model using validation y-AUC.

    Selection uses only downstream labels y. If several C values are effectively tied,
    choose the one with fewer nonzero concept weights for interpretability.
    """
    y_train = to_numpy(y_train).astype(int).reshape(-1)
    y_val = to_numpy(y_val).astype(int).reshape(-1)
    rows = []

    for C in c_grid:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,
                C=C,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-3,
                random_state=0,
            ),
        )
        clf.fit(Z_train, y_train)
        val_prob = clf.predict_proba(Z_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)
        coef = clf.named_steps["logisticregression"].coef_[0]
        n_nonzero = int((np.abs(coef) > 1e-8).sum())
        rows.append({"C": C, "val_task_auc": val_auc, "n_nonzero": n_nonzero, "clf": clf, "coef": coef})

    best_auc = max(row["val_task_auc"] for row in rows)
    eligible = [row for row in rows if row["val_task_auc"] >= best_auc - TASK_HEAD_AUC_TOL]
    best = min(eligible, key=lambda row: (row["n_nonzero"], -row["val_task_auc"], row["C"]))

    coef = best["coef"]
    ranked_direction_ids = np.argsort(-np.abs(coef)).astype(int).tolist()
    return best, pd.DataFrame([{k: v for k, v in row.items() if k not in ["clf", "coef"]} for row in rows]), ranked_direction_ids


def match_selected_directions_on_val(Z_val, hidden_val, selected_direction_ids):
    """For each hidden concept, choose the selected discovered direction with best val AUC."""
    H_val = to_numpy(hidden_val).astype(int)
    selected_scores = Z_val[:, selected_direction_ids]
    auc, directions = residual_hidden_auc_matrix(selected_scores, H_val)
    rows = []

    for h in range(H_val.shape[1]):
        local_idx = int(np.nanargmax(auc[:, h]))
        discovered_idx = int(selected_direction_ids[local_idx])
        rows.append({
            "hidden_idx": h,
            "selected_local_idx": local_idx,
            "discovered_direction_idx": discovered_idx,
            "val_discovery_auc": float(auc[local_idx, h]),
            "direction": directions[local_idx, h],
        })
    return pd.DataFrame(rows)


def evaluate_fixed_direction_matches_on_test(Z_test, hidden_test, matches):
    """Evaluate validation-chosen discovered-direction/hidden matches on held-out test."""
    H_test = to_numpy(hidden_test).astype(int)
    rows = []

    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        j = int(row.discovered_direction_idx)
        score = Z_test[:, j]
        if row.direction == "negative":
            score = -score

        # AUC evaluates ranking quality. Accuracy/F1 use a simple zero threshold
        # because candidate scores are centered by train standardization and projections.
        pred = (score >= 0.0).astype(int)
        rows.append({
            "hidden_idx": h,
            "discovered_direction_idx": j,
            "direction": row.direction,
            "val_discovery_auc": row.val_discovery_auc,
            "discovery_auc": roc_auc_score(H_test[:, h], score),
            "discovery_accuracy_at_zero": accuracy_score(H_test[:, h], pred),
            "discovery_f1_at_zero": f1_score(H_test[:, h], pred, zero_division=0),
        })
    return pd.DataFrame(rows)


# Stage 1: create candidate directions from train residuals only.
X_train_disc, X_val_disc, X_test_disc, disc_mean, disc_std = standardize_residual_splits_for_discovery(
    splits["train"]["res_mu"],
    splits["val"]["res_mu"],
    splits["test"]["res_mu"],
)
candidate_direction_sets = build_residual_direction_candidates(X_train_disc)

# Stage 2/3: for each candidate set, select directions using y, then evaluate hidden recovery.
task_guided_details = []
task_guided_summaries = []
task_head_tuning_tables = {}

for generator_name, directions in candidate_direction_sets.items():
    print("=" * 80)
    print(f"Candidate generator: {generator_name} | directions: {directions.shape[0]}")

    Z_train = project_residual_directions(X_train_disc, directions)
    Z_val = project_residual_directions(X_val_disc, directions)
    Z_test = project_residual_directions(X_test_disc, directions)

    task_head, tuning_table, ranked_direction_ids = fit_sparse_task_head(
        Z_train,
        splits["train"]["y"],
        Z_val,
        splits["val"]["y"],
    )
    task_head_tuning_tables[generator_name] = tuning_table

    test_task_prob = task_head["clf"].predict_proba(Z_test)[:, 1]
    test_task_auc = roc_auc_score(to_numpy(splits["test"]["y"]).astype(int), test_task_prob)
    print(
        f"selected C={task_head['C']} | val y-AUC={task_head['val_task_auc']:.3f} | "
        f"test y-AUC={test_task_auc:.3f} | nonzero directions={task_head['n_nonzero']}"
    )

    # Evaluate each effective top_m once. If there are only 10 candidate
    # directions, requested values 10/20/50 would otherwise all become 10 and
    # create duplicate rows that corrupt top-k summaries.
    effective_top_ms = sorted({min(m, len(ranked_direction_ids)) for m in TASK_DISCOVERY_TOP_MS})
    for top_m in effective_top_ms:
        selected_direction_ids = ranked_direction_ids[:top_m]

        # Hidden labels enter only here, after candidate generation and task selection.
        val_matches = match_selected_directions_on_val(
            Z_val,
            splits["val"]["hidden_residuals"],
            selected_direction_ids,
        )
        test_eval = evaluate_fixed_direction_matches_on_test(
            Z_test,
            splits["test"]["hidden_residuals"],
            val_matches,
        )
        test_eval = attach_relevance(test_eval, relevance).sort_values("rank_abs_w")
        test_eval["generator"] = generator_name
        test_eval["top_m_task_selected_directions"] = top_m
        test_eval["task_head_val_auc"] = task_head["val_task_auc"]
        test_eval["task_head_test_auc"] = test_task_auc
        test_eval["task_head_nonzero_directions"] = task_head["n_nonzero"]
        test_eval["selected_direction_ids"] = [selected_direction_ids] * len(test_eval)
        task_guided_details.append(test_eval)

        top1 = test_eval.head(1)
        top3 = test_eval.head(3)
        top5 = test_eval.head(5)
        task_guided_summaries.append({
            "generator": generator_name,
            "top_m_task_selected_directions": top_m,
            "task_head_val_auc": task_head["val_task_auc"],
            "task_head_test_auc": test_task_auc,
            "task_head_nonzero_directions": task_head["n_nonzero"],
            "top1_mean_discovery_auc": top1["discovery_auc"].mean(),
            "top3_mean_discovery_auc": top3["discovery_auc"].mean(),
            "top5_mean_discovery_auc": top5["discovery_auc"].mean(),
            "top5_unique_matched_directions": int(test_eval.head(5)["discovered_direction_idx"].nunique()),
            "top5_matched_direction_ids": test_eval.head(5)["discovered_direction_idx"].tolist(),
        })

task_guided_recovery_eval = pd.concat(task_guided_details, ignore_index=True)
task_guided_summary = pd.DataFrame(task_guided_summaries).sort_values(
    ["top5_mean_discovery_auc", "top3_mean_discovery_auc", "top5_unique_matched_directions"],
    ascending=[False, False, False],
)

print("Task-guided residual direction discovery summary")
display(task_guided_summary)

best_task_guided = task_guided_summary.iloc[0]
best_task_guided_detail = task_guided_recovery_eval[
    (task_guided_recovery_eval["generator"] == best_task_guided["generator"])
    & (task_guided_recovery_eval["top_m_task_selected_directions"] == best_task_guided["top_m_task_selected_directions"])
].sort_values("rank_abs_w")

print("Best task-guided discovery detail ranked by abs(w_hid)")
display(best_task_guided_detail[[
    "generator",
    "top_m_task_selected_directions",
    "hidden_idx",
    "discovered_direction_idx",
    "discovery_auc",
    "val_discovery_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "task_head_test_auc",
]])

# Compare the best task-guided discovery result to existing baselines.
task_guided_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(best_task_guided_detail[["hidden_idx", "discovery_auc", "discovered_direction_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Raw axis vs dense probe vs best task-guided discovery")
display(task_guided_comparison)

best_task_guided_summary = pd.DataFrame({
    "method": ["raw_axis", "dense_probe", "task_guided_discovery"],
    "top1_mean_auc": [
        task_guided_comparison.head(1)["axis_auc"].mean(),
        task_guided_comparison.head(1)["distributed_auc"].mean(),
        task_guided_comparison.head(1)["discovery_auc"].mean(),
    ],
    "top3_mean_auc": [
        task_guided_comparison.head(3)["axis_auc"].mean(),
        task_guided_comparison.head(3)["distributed_auc"].mean(),
        task_guided_comparison.head(3)["discovery_auc"].mean(),
    ],
    "top5_mean_auc": [
        task_guided_comparison.head(5)["axis_auc"].mean(),
        task_guided_comparison.head(5)["distributed_auc"].mean(),
        task_guided_comparison.head(5)["discovery_auc"].mean(),
    ],
})
display(best_task_guided_summary)


Candidate generator: raw_axes | directions: 10
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero directions=9
Candidate generator: pca | directions: 10
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero directions=1
Candidate generator: sparse_pca | directions: 10
selected C=0.003 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero directions=8
Candidate generator: combined_raw_pca_sparsepca | directions: 30
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero directions=7
Task-guided residual direction discovery summary


,generator,top_m_task_selected_directions,task_head_val_auc,task_head_test_auc,task_head_nonzero_directions,top1_mean_discovery_auc,top3_mean_discovery_auc,top5_mean_discovery_auc,top5_unique_matched_directions,top5_matched_direction_ids
5,sparse_pca,10,0.894186,0.894142,8,0.700518,0.701425,0.650059,4,"[7, 8, 5, 3, 7]"
7,combined_raw_pca_sparsepca,10,0.894252,0.894051,7,0.700518,0.701425,0.650059,4,"[27, 28, 1, 0, 27]"
8,combined_raw_pca_sparsepca,20,0.894252,0.894051,7,0.700518,0.701425,0.650059,4,"[27, 28, 1, 0, 27]"
9,combined_raw_pca_sparsepca,30,0.894252,0.894051,7,0.700518,0.701425,0.650059,4,"[27, 28, 1, 0, 27]"
0,raw_axes,5,0.894192,0.894179,9,0.700518,0.701371,0.650027,4,"[3, 6, 1, 0, 3]"
1,raw_axes,10,0.894192,0.894179,9,0.700518,0.701371,0.650027,4,"[3, 6, 1, 0, 3]"
4,sparse_pca,5,0.894186,0.894142,8,0.699438,0.701064,0.649281,4,"[4, 8, 5, 3, 5]"
6,combined_raw_pca_sparsepca,5,0.894252,0.894051,7,0.695502,0.699753,0.648494,4,"[10, 28, 1, 0, 1]"
2,pca,5,0.894231,0.893791,1,0.695502,0.696879,0.646126,1,"[0, 0, 0, 0, 0]"
3,pca,10,0.894231,0.893791,1,0.695502,0.696879,0.646126,1,"[0, 0, 0, 0, 0]"


Best task-guided discovery detail ranked by abs(w_hid)


,generator,top_m_task_selected_directions,hidden_idx,discovered_direction_idx,discovery_auc,val_discovery_auc,w_hid,abs_w_hid,rank_abs_w,task_head_test_auc
100,sparse_pca,10,18,7,0.700518,0.694672,-0.705692,0.705692,1,0.894142
101,sparse_pca,10,2,8,0.732201,0.740728,-0.648612,0.648612,2,0.894142
102,sparse_pca,10,9,5,0.671554,0.651557,0.230210,0.230210,3,0.894142
103,sparse_pca,10,19,3,0.544257,0.546242,-0.150507,0.150507,4,0.894142
104,sparse_pca,10,7,7,0.601763,0.606747,0.075182,0.075182,5,0.894142
117,sparse_pca,10,17,7,0.570079,0.560857,0.000000,0.000000,6,0.894142
116,sparse_pca,10,16,4,0.573417,0.559573,0.000000,0.000000,6,0.894142
115,sparse_pca,10,15,4,0.522905,0.519063,0.000000,0.000000,6,0.894142
114,sparse_pca,10,14,7,0.532370,0.520966,0.000000,0.000000,6,0.894142
113,sparse_pca,10,13,7,0.627357,0.618274,0.000000,0.000000,6,0.894142


Raw axis vs dense probe vs best task-guided discovery


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,discovery_auc,discovered_direction_idx
0,18,0.700518,0.705692,1,0.713136,0.700518,7
1,2,0.732040,0.648612,2,0.746453,0.732201,8
2,9,0.671554,0.230210,3,0.685063,0.671554,5
3,19,0.544257,0.150507,4,0.571436,0.544257,3
4,7,0.601763,0.075182,5,0.618079,0.601763,7
17,17,0.570079,0.000000,6,0.583489,0.570079,7
16,16,0.572152,0.000000,6,0.589729,0.573417,4
15,15,0.522905,0.000000,6,0.594516,0.522905,4
14,14,0.533031,0.000000,6,0.573579,0.532370,7
13,13,0.627357,0.000000,6,0.647779,0.627357,7


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701371,0.650027
1,dense_probe,0.713136,0.714884,0.666833
2,task_guided_discovery,0.700518,0.701425,0.650059


## Concept-Conditioned Task-Guided Residual Direction Discovery

This variant selects residual directions for their incremental task value after the task head already sees the observed concepts. The sparse head is trained on `[concepts, residual_direction_scores]`, but residual direction ranking uses only the residual coefficients.


In [380]:
# -----------------------------
# Concept-conditioned task-guided residual direction discovery
# -----------------------------
# The previous task-guided section selected residual directions using y alone.
# That can favor broad global task directions. Here we control for the known
# concept bottleneck: the sparse task head sees both observed concepts and
# candidate residual directions, and we rank only the residual-direction weights.
#
# Hidden labels are still used only after selection, for validation matching and
# held-out test evaluation.

CONDITIONED_TASK_HEAD_C_GRID = TASK_HEAD_C_GRID if "TASK_HEAD_C_GRID" in globals() else [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
CONDITIONED_TASK_TOP_MS = TASK_DISCOVERY_TOP_MS if "TASK_DISCOVERY_TOP_MS" in globals() else [5, 10, 20, 50]
CONDITIONED_TASK_HEAD_AUC_TOL = TASK_HEAD_AUC_TOL if "TASK_HEAD_AUC_TOL" in globals() else 0.005


def get_concept_conditioning_source(splits):
    """Choose the concept representation used as conditioning input.

    Prefer model-predicted concept means (`c_mu`) when the joint concept/residual
    channel was saved. Fall back to predicted probabilities/sample means if present,
    and only use ground-truth dataset concepts as an oracle fallback.
    """
    preferred_keys = ["c_mu", "concept_probs", "concept_sample_mean", "concepts"]
    for key in preferred_keys:
        if all(key in splits[split] for split in ["train", "val", "test"]):
            return key
    raise KeyError(f"No concept conditioning source found. Tried: {preferred_keys}")


def standardize_concepts_for_conditioning(train_concepts, val_concepts, test_concepts):
    """Standardize concept inputs using train statistics only."""
    C_train = to_numpy(train_concepts).astype(float)
    C_val = to_numpy(val_concepts).astype(float)
    C_test = to_numpy(test_concepts).astype(float)
    mean = C_train.mean(axis=0, keepdims=True)
    std = C_train.std(axis=0, keepdims=True) + 1e-8
    return (C_train - mean) / std, (C_val - mean) / std, (C_test - mean) / std


def fit_concept_conditioned_sparse_task_head(
    C_train,
    Z_train,
    y_train,
    C_val,
    Z_val,
    y_val,
    c_grid=CONDITIONED_TASK_HEAD_C_GRID,
):
    """
    Train sparse task heads on [concepts, residual-direction scores].

    Model selection uses validation y-AUC. Residual direction ranking uses only
    coefficients attached to Z, so observed-concept coefficients do not directly
    become discovered residual concepts.
    """
    y_train = to_numpy(y_train).astype(int).reshape(-1)
    y_val = to_numpy(y_val).astype(int).reshape(-1)
    X_train = np.concatenate([C_train, Z_train], axis=1)
    X_val = np.concatenate([C_val, Z_val], axis=1)
    n_concepts = C_train.shape[1]

    rows = []
    for C in c_grid:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,
                C=C,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-3,
                random_state=0,
            ),
        )
        clf.fit(X_train, y_train)
        val_prob = clf.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)

        coef = clf.named_steps["logisticregression"].coef_[0]
        concept_coef = coef[:n_concepts]
        residual_coef = coef[n_concepts:]
        rows.append({
            "C": C,
            "val_task_auc": val_auc,
            "n_nonzero_total": int((np.abs(coef) > 1e-8).sum()),
            "n_nonzero_concepts": int((np.abs(concept_coef) > 1e-8).sum()),
            "n_nonzero_residual_directions": int((np.abs(residual_coef) > 1e-8).sum()),
            "clf": clf,
            "coef": coef,
            "residual_coef": residual_coef,
        })

    best_auc = max(row["val_task_auc"] for row in rows)
    eligible = [row for row in rows if row["val_task_auc"] >= best_auc - CONDITIONED_TASK_HEAD_AUC_TOL]
    best = min(eligible, key=lambda row: (row["n_nonzero_residual_directions"], row["n_nonzero_total"], -row["val_task_auc"], row["C"]))

    residual_coef = best["residual_coef"]
    ranked_direction_ids = np.argsort(-np.abs(residual_coef)).astype(int).tolist()
    nonzero_ranked_direction_ids = [idx for idx in ranked_direction_ids if abs(residual_coef[idx]) > 1e-8]
    best["nonzero_ranked_direction_ids"] = nonzero_ranked_direction_ids
    tuning = pd.DataFrame([{k: v for k, v in row.items() if k not in ["clf", "coef", "residual_coef"]} for row in rows])
    return best, tuning, ranked_direction_ids


# Reuse the candidate directions from the task-guided discovery section when available;
# otherwise build them here from train residuals.
if "candidate_direction_sets" not in globals():
    X_train_disc, X_val_disc, X_test_disc, disc_mean, disc_std = standardize_residual_splits_for_discovery(
        splits["train"]["res_mu"],
        splits["val"]["res_mu"],
        splits["test"]["res_mu"],
    )
    candidate_direction_sets = build_residual_direction_candidates(X_train_disc)



concept_conditioning_key = get_concept_conditioning_source(splits)
print(f"Conditioning task head on concept source: {concept_conditioning_key}")

C_train_cond, C_val_cond, C_test_cond = standardize_concepts_for_conditioning(
    splits["train"][concept_conditioning_key],
    splits["val"][concept_conditioning_key],
    splits["test"][concept_conditioning_key],
)

conditioned_task_guided_details = []
conditioned_task_guided_summaries = []
conditioned_task_head_tuning_tables = {}

for generator_name, directions in candidate_direction_sets.items():
    print("=" * 80)
    print(f"Concept-conditioned generator: {generator_name} | residual directions: {directions.shape[0]}")

    Z_train = project_residual_directions(X_train_disc, directions)
    Z_val = project_residual_directions(X_val_disc, directions)
    Z_test = project_residual_directions(X_test_disc, directions)

    task_head, tuning_table, ranked_direction_ids = fit_concept_conditioned_sparse_task_head(
        C_train_cond,
        Z_train,
        splits["train"]["y"],
        C_val_cond,
        Z_val,
        splits["val"]["y"],
    )
    conditioned_task_head_tuning_tables[generator_name] = tuning_table

    X_test_task = np.concatenate([C_test_cond, Z_test], axis=1)
    test_task_prob = task_head["clf"].predict_proba(X_test_task)[:, 1]
    test_task_auc = roc_auc_score(to_numpy(splits["test"]["y"]).astype(int), test_task_prob)
    print(
        f"selected C={task_head['C']} | val y-AUC={task_head['val_task_auc']:.3f} | "
        f"test y-AUC={test_task_auc:.3f} | nonzero concepts={task_head['n_nonzero_concepts']} | "
        f"nonzero residual dirs={task_head['n_nonzero_residual_directions']}"
    )

    selectable_direction_ids = task_head["nonzero_ranked_direction_ids"]
    if len(selectable_direction_ids) == 0:
        # If the sparse task head assigns no residual weight, there is no
        # concept-conditioned residual discovery to evaluate for this generator.
        print("No nonzero residual directions selected by the conditioned task head; skipping hidden recovery.")
        continue

    # Evaluate each effective top_m once. This avoids duplicate rows when
    # top_m requests exceed the number of nonzero residual directions selected
    # by the sparse conditioned task head.
    effective_top_ms = sorted({min(m, len(selectable_direction_ids)) for m in CONDITIONED_TASK_TOP_MS})
    for top_m in effective_top_ms:
        selected_direction_ids = selectable_direction_ids[:top_m]

        val_matches = match_selected_directions_on_val(
            Z_val,
            splits["val"]["hidden_residuals"],
            selected_direction_ids,
        )
        test_eval = evaluate_fixed_direction_matches_on_test(
            Z_test,
            splits["test"]["hidden_residuals"],
            val_matches,
        )
        test_eval = attach_relevance(test_eval, relevance).sort_values("rank_abs_w")
        test_eval["generator"] = generator_name
        test_eval["top_m_task_selected_directions"] = top_m
        test_eval["conditioned_task_head_val_auc"] = task_head["val_task_auc"]
        test_eval["conditioned_task_head_test_auc"] = test_task_auc
        test_eval["concept_conditioning_source"] = concept_conditioning_key
        test_eval["conditioned_nonzero_concepts"] = task_head["n_nonzero_concepts"]
        test_eval["conditioned_nonzero_residual_directions"] = task_head["n_nonzero_residual_directions"]
        test_eval["selected_direction_ids"] = [selected_direction_ids] * len(test_eval)
        conditioned_task_guided_details.append(test_eval)

        top1 = test_eval.head(1)
        top3 = test_eval.head(3)
        top5 = test_eval.head(5)
        conditioned_task_guided_summaries.append({
            "generator": generator_name,
            "top_m_task_selected_directions": top_m,
            "conditioned_task_head_val_auc": task_head["val_task_auc"],
            "conditioned_task_head_test_auc": test_task_auc,
            "concept_conditioning_source": concept_conditioning_key,
            "conditioned_nonzero_concepts": task_head["n_nonzero_concepts"],
            "conditioned_nonzero_residual_directions": task_head["n_nonzero_residual_directions"],
            "top1_mean_discovery_auc": top1["discovery_auc"].mean(),
            "top3_mean_discovery_auc": top3["discovery_auc"].mean(),
            "top5_mean_discovery_auc": top5["discovery_auc"].mean(),
            "top5_unique_matched_directions": int(test_eval.head(5)["discovered_direction_idx"].nunique()),
            "top5_matched_direction_ids": test_eval.head(5)["discovered_direction_idx"].tolist(),
        })

conditioned_task_guided_recovery_eval = pd.concat(conditioned_task_guided_details, ignore_index=True)
conditioned_task_guided_summary = pd.DataFrame(conditioned_task_guided_summaries).sort_values(
    ["top5_mean_discovery_auc", "top3_mean_discovery_auc", "top5_unique_matched_directions"],
    ascending=[False, False, False],
)

print("Concept-conditioned task-guided residual direction discovery summary")
display(conditioned_task_guided_summary)

best_conditioned_task_guided = conditioned_task_guided_summary.iloc[0]
best_conditioned_task_guided_detail = conditioned_task_guided_recovery_eval[
    (conditioned_task_guided_recovery_eval["generator"] == best_conditioned_task_guided["generator"])
    & (
        conditioned_task_guided_recovery_eval["top_m_task_selected_directions"]
        == best_conditioned_task_guided["top_m_task_selected_directions"]
    )
].sort_values("rank_abs_w")

print("Best concept-conditioned task-guided discovery detail ranked by abs(w_hid)")
display(best_conditioned_task_guided_detail[[
    "generator",
    "top_m_task_selected_directions",
    "hidden_idx",
    "discovered_direction_idx",
    "discovery_auc",
    "val_discovery_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "conditioned_task_head_test_auc",
    "concept_conditioning_source",
    "conditioned_nonzero_concepts",
    "conditioned_nonzero_residual_directions",
]])

conditioned_task_guided_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(
        best_conditioned_task_guided_detail[["hidden_idx", "discovery_auc", "discovered_direction_idx"]],
        on="hidden_idx",
        how="left",
    )
    .sort_values("rank_abs_w")
)

print("Raw axis vs dense probe vs best concept-conditioned task-guided discovery")
display(conditioned_task_guided_comparison)

best_conditioned_task_guided_summary = pd.DataFrame({
    "method": ["raw_axis", "dense_probe", "concept_conditioned_task_guided"],
    "top1_mean_auc": [
        conditioned_task_guided_comparison.head(1)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(1)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(1)["discovery_auc"].mean(),
    ],
    "top3_mean_auc": [
        conditioned_task_guided_comparison.head(3)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(3)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(3)["discovery_auc"].mean(),
    ],
    "top5_mean_auc": [
        conditioned_task_guided_comparison.head(5)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(5)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(5)["discovery_auc"].mean(),
    ],
})
display(best_conditioned_task_guided_summary)


Conditioning task head on concept source: c_mu
Concept-conditioned generator: raw_axes | residual directions: 10
selected C=0.003 | val y-AUC=0.894 | test y-AUC=0.895 | nonzero concepts=6 | nonzero residual dirs=8
Concept-conditioned generator: pca | residual directions: 10
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=3 | nonzero residual dirs=1
Concept-conditioned generator: sparse_pca | residual directions: 10
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=5 | nonzero residual dirs=7
Concept-conditioned generator: combined_raw_pca_sparsepca | residual directions: 30
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=3 | nonzero residual dirs=7
Concept-conditioned task-guided residual direction discovery summary


,generator,top_m_task_selected_directions,conditioned_task_head_val_auc,conditioned_task_head_test_auc,concept_conditioning_source,conditioned_nonzero_concepts,conditioned_nonzero_residual_directions,top1_mean_discovery_auc,top3_mean_discovery_auc,top5_mean_discovery_auc,top5_unique_matched_directions,top5_matched_direction_ids
4,sparse_pca,7,0.894261,0.894498,c_mu,5,7,0.700518,0.701425,0.650059,4,"[7, 8, 5, 3, 7]"
1,raw_axes,8,0.894314,0.894519,c_mu,6,8,0.700518,0.701371,0.650027,4,"[3, 6, 1, 0, 3]"
3,sparse_pca,5,0.894261,0.894498,c_mu,5,7,0.695617,0.699791,0.648517,4,"[0, 8, 5, 3, 5]"
6,combined_raw_pca_sparsepca,7,0.894321,0.894355,c_mu,3,7,0.695502,0.699753,0.648494,4,"[10, 28, 1, 0, 1]"
0,raw_axes,5,0.894314,0.894519,c_mu,6,8,0.694508,0.699368,0.648263,4,"[7, 6, 1, 0, 1]"
5,combined_raw_pca_sparsepca,5,0.894321,0.894355,c_mu,3,7,0.695502,0.698441,0.647396,4,"[10, 28, 20, 0, 10]"
2,pca,1,0.894326,0.894231,c_mu,3,1,0.695502,0.696879,0.646126,1,"[0, 0, 0, 0, 0]"


Best concept-conditioned task-guided discovery detail ranked by abs(w_hid)


,generator,top_m_task_selected_directions,hidden_idx,discovered_direction_idx,discovery_auc,val_discovery_auc,w_hid,abs_w_hid,rank_abs_w,conditioned_task_head_test_auc,concept_conditioning_source,conditioned_nonzero_concepts,conditioned_nonzero_residual_directions
80,sparse_pca,7,18,7,0.700518,0.694672,-0.705692,0.705692,1,0.894498,c_mu,5,7
81,sparse_pca,7,2,8,0.732201,0.740728,-0.648612,0.648612,2,0.894498,c_mu,5,7
82,sparse_pca,7,9,5,0.671554,0.651557,0.230210,0.230210,3,0.894498,c_mu,5,7
83,sparse_pca,7,19,3,0.544257,0.546242,-0.150507,0.150507,4,0.894498,c_mu,5,7
84,sparse_pca,7,7,7,0.601763,0.606747,0.075182,0.075182,5,0.894498,c_mu,5,7
97,sparse_pca,7,17,7,0.570079,0.560857,0.000000,0.000000,6,0.894498,c_mu,5,7
96,sparse_pca,7,16,4,0.573417,0.559573,0.000000,0.000000,6,0.894498,c_mu,5,7
95,sparse_pca,7,15,4,0.522905,0.519063,0.000000,0.000000,6,0.894498,c_mu,5,7
94,sparse_pca,7,14,7,0.532370,0.520966,0.000000,0.000000,6,0.894498,c_mu,5,7
93,sparse_pca,7,13,7,0.627357,0.618274,0.000000,0.000000,6,0.894498,c_mu,5,7


Raw axis vs dense probe vs best concept-conditioned task-guided discovery


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,discovery_auc,discovered_direction_idx
0,18,0.700518,0.705692,1,0.713136,0.700518,7
1,2,0.732040,0.648612,2,0.746453,0.732201,8
2,9,0.671554,0.230210,3,0.685063,0.671554,5
3,19,0.544257,0.150507,4,0.571436,0.544257,3
4,7,0.601763,0.075182,5,0.618079,0.601763,7
17,17,0.570079,0.000000,6,0.583489,0.570079,7
16,16,0.572152,0.000000,6,0.589729,0.573417,4
15,15,0.522905,0.000000,6,0.594516,0.522905,4
14,14,0.533031,0.000000,6,0.573579,0.532370,7
13,13,0.627357,0.000000,6,0.647779,0.627357,7


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701371,0.650027
1,dense_probe,0.713136,0.714884,0.666833
2,concept_conditioned_task_guided,0.700518,0.701425,0.650059


## Sparse Probe Recovery

This supervised diagnostic asks whether each hidden concept is recoverable from a small linear combination of residual dimensions. It is not an unsupervised discovery method; it tells us whether the information is sparse-linear, dense-linear, or mostly absent.


In [381]:
# -----------------------------
# Sparse probe recovery
# -----------------------------
# Raw-axis recovery asks whether one residual dimension recovers a hidden concept.
# Dense probes ask whether the whole residual vector linearly recovers it.
# L1 sparse probes sit between those: can a small subset of residual dimensions
# recover the hidden concept?
#
# Important: this is supervised. Hidden labels are used to train the probes, so
# this is a diagnostic of recoverability, not an unsupervised discovery method.

L1_PROBE_C_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
# Treat probes within this validation-AUC band as effectively tied, then prefer sparsity.
# This keeps the diagnostic honest: a 50-dimensional L1 probe is just the dense probe wearing a hat.
L1_PROBE_AUC_TOL = 0.01
L1_COEF_TOL = 1e-6


def fit_l1_probe_for_hidden(X_train, y_train, X_val, y_val, X_test, y_test, c_grid=L1_PROBE_C_GRID):
    """Tune an L1 logistic probe on validation and evaluate the selected model on test."""
    rows = []
    best = None

    for C in c_grid:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,
                C=C,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-3,
                random_state=0,
            ),
        )
        clf.fit(X_train, y_train)

        val_prob = clf.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)
        coef = clf.named_steps["logisticregression"].coef_[0]
        n_nonzero = int((np.abs(coef) > L1_COEF_TOL).sum())

        row = {
            "C": C,
            "val_auc": val_auc,
            "n_nonzero": n_nonzero,
            "clf": clf,
            "coef": coef,
        }
        rows.append(row)

    # First find the best validation AUC. Then select the sparsest model that is
    # close to that best score. This reports the smallest residual subset that
    # retains essentially the same recoverability as the best L1 probe.
    best_val_auc = max(row["val_auc"] for row in rows)
    eligible = [row for row in rows if row["val_auc"] >= best_val_auc - L1_PROBE_AUC_TOL]
    best = min(eligible, key=lambda row: (row["n_nonzero"], -row["val_auc"], row["C"]))

    clf = best["clf"]
    test_prob = clf.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)
    coef = best["coef"]
    selected_dims = np.flatnonzero(np.abs(coef) > L1_COEF_TOL).tolist()
    coef_order = np.argsort(-np.abs(coef))
    top_coef_dims = [int(dim) for dim in coef_order[: min(10, len(coef_order))] if abs(coef[dim]) > L1_COEF_TOL]

    return {
        "best_C": best["C"],
        "best_val_auc_seen": best_val_auc,
        "selected_val_auc": best["val_auc"],
        "sparse_auc": roc_auc_score(y_test, test_prob),
        "sparse_accuracy": accuracy_score(y_test, test_pred),
        "sparse_f1": f1_score(y_test, test_pred, zero_division=0),
        "n_nonzero_dims": len(selected_dims),
        "selected_dims": selected_dims,
        "top_coef_dims": top_coef_dims,
        "coef_l1_norm": float(np.abs(coef).sum()),
        "coef_l2_norm": float(np.sqrt((coef ** 2).sum())),
    }


def sparse_probe_train_val_test(train_scores, train_hidden, val_scores, val_hidden, test_scores, test_hidden):
    X_train = to_numpy(train_scores).astype(float)
    X_val = to_numpy(val_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_val = to_numpy(val_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_val[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        result = fit_l1_probe_for_hidden(
            X_train,
            H_train[:, h],
            X_val,
            H_val[:, h],
            X_test,
            H_test[:, h],
        )
        rows.append({"hidden_idx": h, **result})

    return pd.DataFrame(rows)


sparse_probe_eval = sparse_probe_train_val_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["val"]["res_mu"],
    splits["val"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)
sparse_probe_recovery_eval = add_task_relevance(sparse_probe_eval, splits["test"]["w_hid"]).sort_values("rank_abs_w")


print("Sparse L1 probe recovery ranked by abs(w_hid)")
display(sparse_probe_recovery_eval[[
    "hidden_idx",
    "sparse_auc",
    "sparse_accuracy",
    "sparse_f1",
    "best_C",
    "best_val_auc_seen",
    "selected_val_auc",
    "n_nonzero_dims",
    "top_coef_dims",
    "selected_dims",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
]])

print("Sparse L1 probe recovery among top-k hidden concepts by abs(w_hid)")
display(summarize_recovery_by_relevance(sparse_probe_recovery_eval, "sparse_auc", "abs_w_hid"))

# Compare the ladder of recoverability:
# raw_axis: one residual dimension
# sparse_l1_probe: sparse supervised linear combination
# dense_probe: dense supervised linear combination
probe_ladder = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(sparse_probe_recovery_eval[["hidden_idx", "sparse_auc", "n_nonzero_dims", "top_coef_dims", "selected_dims"]], on="hidden_idx", how="left")
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Recoverability ladder: raw axis vs sparse L1 probe vs dense probe")
display(probe_ladder)

probe_ladder_summary = pd.DataFrame({
    "method": ["raw_axis", "sparse_l1_probe", "dense_probe"],
    "top1_mean_auc": [
        probe_ladder.head(1)["axis_auc"].mean(),
        probe_ladder.head(1)["sparse_auc"].mean(),
        probe_ladder.head(1)["distributed_auc"].mean(),
    ],
    "top3_mean_auc": [
        probe_ladder.head(3)["axis_auc"].mean(),
        probe_ladder.head(3)["sparse_auc"].mean(),
        probe_ladder.head(3)["distributed_auc"].mean(),
    ],
    "top5_mean_auc": [
        probe_ladder.head(5)["axis_auc"].mean(),
        probe_ladder.head(5)["sparse_auc"].mean(),
        probe_ladder.head(5)["distributed_auc"].mean(),
    ],
    "spearman_abs_w_auc": [
        probe_ladder[["abs_w_hid", "axis_auc"]].corr(method="spearman").iloc[0, 1],
        probe_ladder[["abs_w_hid", "sparse_auc"]].corr(method="spearman").iloc[0, 1],
        probe_ladder[["abs_w_hid", "distributed_auc"]].corr(method="spearman").iloc[0, 1],
    ],
})
display(probe_ladder_summary)

print("Top task-relevant sparse probes: selected residual dimensions")
top_sparse_dims = probe_ladder.head(5)[["hidden_idx", "abs_w_hid", "sparse_auc", "n_nonzero_dims", "top_coef_dims", "selected_dims"]]
display(top_sparse_dims)


Sparse L1 probe recovery ranked by abs(w_hid)


,hidden_idx,sparse_auc,sparse_accuracy,sparse_f1,best_C,best_val_auc_seen,selected_val_auc,n_nonzero_dims,top_coef_dims,selected_dims,w_hid,abs_w_hid,rank_abs_w
18,18,0.705682,0.6562,0.651672,0.0100,0.705810,0.699672,7,"[3, 9, 6, 0, 5, 8, 2]","[0, 2, 3, 5, 6, 8, 9]",-0.705692,0.705692,1
2,2,0.745279,0.6793,0.672722,0.0300,0.755446,0.754469,9,"[6, 3, 8, 9, 4, 0, 7, 2, 1]","[0, 1, 2, 3, 4, 6, 7, 8, 9]",-0.648612,0.648612,2
9,9,0.676411,0.6234,0.631867,0.0100,0.664818,0.656081,4,"[3, 0, 1, 7]","[0, 1, 3, 7]",0.230210,0.230210,3
19,19,0.569333,0.5451,0.535579,0.0300,0.572558,0.569774,8,"[9, 6, 0, 8, 3, 2, 7, 4]","[0, 2, 3, 4, 6, 7, 8, 9]",-0.150507,0.150507,4
7,7,0.613665,0.5840,0.579118,0.0300,0.619741,0.617009,8,"[3, 7, 0, 1, 4, 5, 6, 9]","[0, 1, 3, 4, 5, 6, 7, 9]",0.075182,0.075182,5
4,4,0.627355,0.5918,0.581762,0.0010,0.624963,0.616884,2,"[1, 3]","[1, 3]",0.000000,0.000000,6
5,5,0.647120,0.6033,0.596645,0.0003,0.642122,0.640910,1,[7],[7],0.000000,0.000000,6
6,6,0.643986,0.6039,0.604809,0.0300,0.638337,0.635257,9,"[3, 8, 0, 9, 7, 6, 4, 5, 1]","[0, 1, 3, 4, 5, 6, 7, 8, 9]",0.000000,0.000000,6
8,8,0.604143,0.5749,0.579650,3.0000,0.608464,0.608464,10,"[9, 8, 1, 4, 2, 6, 0, 5, 7, 3]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",0.000000,0.000000,6
1,1,0.560319,0.5430,0.551081,0.0300,0.564263,0.562388,8,"[7, 6, 3, 8, 5, 2, 4, 1]","[1, 2, 3, 4, 5, 6, 7, 8]",0.000000,0.000000,6


Sparse L1 probe recovery among top-k hidden concepts by abs(w_hid)


,top_k_by,k,mean_sparse_auc,max_sparse_auc,num_sparse_auc_ge_0_7,hidden_indices
0,abs_w_hid,1,0.705682,0.705682,1,[18]
1,abs_w_hid,3,0.709124,0.745279,2,"[18, 2, 9]"
2,abs_w_hid,5,0.662074,0.745279,2,"[18, 2, 9, 19, 7]"
3,abs_w_hid,10,0.622473,0.745279,2,"[18, 2, 9, 19, 7, 12, 3, 17, 16, 15]"


Recoverability ladder: raw axis vs sparse L1 probe vs dense probe


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,sparse_auc,n_nonzero_dims,top_coef_dims,selected_dims,distributed_auc
0,18,0.700518,0.705692,1,0.705682,7,"[3, 9, 6, 0, 5, 8, 2]","[0, 2, 3, 5, 6, 8, 9]",0.713136
1,2,0.732040,0.648612,2,0.745279,9,"[6, 3, 8, 9, 4, 0, 7, 2, 1]","[0, 1, 2, 3, 4, 6, 7, 8, 9]",0.746453
2,9,0.671554,0.230210,3,0.676411,4,"[3, 0, 1, 7]","[0, 1, 3, 7]",0.685063
3,19,0.544257,0.150507,4,0.569333,8,"[9, 6, 0, 8, 3, 2, 7, 4]","[0, 2, 3, 4, 6, 7, 8, 9]",0.571436
4,7,0.601763,0.075182,5,0.613665,8,"[3, 7, 0, 1, 4, 5, 6, 9]","[0, 1, 3, 4, 5, 6, 7, 9]",0.618079
17,17,0.570079,0.000000,6,0.581498,8,"[6, 3, 1, 7, 5, 0, 4, 8]","[0, 1, 3, 4, 5, 6, 7, 8]",0.583489
16,16,0.572152,0.000000,6,0.585274,8,"[9, 7, 8, 6, 2, 5, 4, 0]","[0, 2, 4, 5, 6, 7, 8, 9]",0.589729
15,15,0.522905,0.000000,6,0.592072,9,"[9, 3, 6, 2, 4, 8, 1, 7, 0]","[0, 1, 2, 3, 4, 6, 7, 8, 9]",0.594516
14,14,0.533031,0.000000,6,0.573588,10,"[6, 1, 3, 9, 5, 7, 4, 8, 2, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",0.573579
13,13,0.627357,0.000000,6,0.647988,10,"[8, 3, 9, 2, 5, 6, 4, 1, 7, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",0.647779


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc,spearman_abs_w_auc
0,raw_axis,0.700518,0.701371,0.650027,0.569182
1,sparse_l1_probe,0.705682,0.709124,0.662074,0.464437
2,dense_probe,0.713136,0.714884,0.666833,0.480247


Top task-relevant sparse probes: selected residual dimensions


,hidden_idx,abs_w_hid,sparse_auc,n_nonzero_dims,top_coef_dims,selected_dims
0,18,0.705692,0.705682,7,"[3, 9, 6, 0, 5, 8, 2]","[0, 2, 3, 5, 6, 8, 9]"
1,2,0.648612,0.745279,9,"[6, 3, 8, 9, 4, 0, 7, 2, 1]","[0, 1, 2, 3, 4, 6, 7, 8, 9]"
2,9,0.230210,0.676411,4,"[3, 0, 1, 7]","[0, 1, 3, 7]"
3,19,0.150507,0.569333,8,"[9, 6, 0, 8, 3, 2, 7, 4]","[0, 2, 3, 4, 6, 7, 8, 9]"
4,7,0.075182,0.613665,8,"[3, 7, 0, 1, 4, 5, 6, 9]","[0, 1, 3, 4, 5, 6, 7, 9]"


## SAE Residual Concept Discovery

This section adapts the BatchTopK SAE idea from the CEM concept-discovery repo to the Residual SCBM residual channel. The SAE is trained unsupervised on `train/res_mu`; validation chooses SAE-feature/hidden-concept matches; test evaluates those fixed matches.


In [382]:
# -----------------------------
# SAE configuration
# -----------------------------
# The SAE is trained only on residual-channel representations (`res_mu`).
# It does not see hidden concept labels during training.
SAE_CONFIG = {
    # The SAE dictionary is overcomplete: for an R-dimensional residual channel,
    # the SAE has dict_size_multiplier * R candidate sparse features.
    # For R=50 and multiplier=4, this gives 200 possible discovered features.
    "dict_size_multiplier": 4,

    # BatchTopK sparsity: across each batch, keep top_k active features per
    # example on average. Smaller values force more sparse/competitive features.
    "top_k": 4,

    # Optimization hyperparameters.
    "batch_size": 2048,
    "n_epochs": 50,
    "lr": 1e-3,
    "beta1": 0.9,
    "beta2": 0.999,
    "l1_coeff": 0.0,
    "max_grad_norm": 1.0,
    "seed": 0,

    # After training, discard SAE features that almost never activate on train.
    "min_train_active_rate": 1e-4,
}


class ResidualBatchTopKSAE(nn.Module):
    """
    BatchTopK sparse autoencoder for residual-channel concept discovery.

    This mirrors the SAE idea used in the CEM concept-discovery repo: learn an
    overcomplete dictionary that reconstructs the representation, while forcing
    only a small number of features to activate. Nonzero SAE activations are then
    treated as candidate discovered residual concepts.
    """

    def __init__(self, input_dim, dict_size, top_k):
        super().__init__()
        self.input_dim = input_dim
        self.dict_size = dict_size
        self.top_k = top_k

        # Decoder bias is the reconstruction baseline. Encoder bias shifts SAE
        # feature thresholds. W_enc maps residual vectors to feature activations;
        # W_dec maps sparse feature activations back to residual space.
        self.b_dec = nn.Parameter(torch.zeros(input_dim))
        self.b_enc = nn.Parameter(torch.zeros(dict_size))
        self.W_enc = nn.Parameter(torch.empty(input_dim, dict_size))
        self.W_dec = nn.Parameter(torch.empty(dict_size, input_dim))

        # Initialize decoder as tied to encoder transpose, then normalize decoder
        # atoms so feature scale is controlled by activations rather than weights.
        nn.init.kaiming_uniform_(self.W_enc)
        self.W_dec.data[:] = self.W_enc.t().data
        self.renorm_decoder_weights()

    @torch.no_grad()
    def renorm_decoder_weights(self):
        # Keep each decoder feature vector at unit norm after optimizer updates.
        self.W_dec.data = self.W_dec.data / (self.W_dec.data.norm(dim=-1, keepdim=True) + 1e-8)

    def encode_dense(self, x):
        # Dense nonnegative feature activations before sparsification.
        return F.relu((x - self.b_dec) @ self.W_enc + self.b_enc)

    def sparsify(self, acts):
        # BatchTopK sparsification. Instead of keeping top_k per row, this keeps
        # top_k * batch_size activations over the entire batch. This creates
        # competition across both features and examples.
        k_total = min(self.top_k * acts.shape[0], acts.numel())
        top = torch.topk(acts.flatten(), k_total, dim=-1)
        return torch.zeros_like(acts.flatten()).scatter(-1, top.indices, top.values).reshape_as(acts)

    def forward(self, x):
        # Encode -> sparsify -> reconstruct. Training minimizes reconstruction
        # error only; no hidden concept labels are used here.
        acts_dense = self.encode_dense(x)
        acts_sparse = self.sparsify(acts_dense)
        x_hat = acts_sparse @ self.W_dec + self.b_dec
        return x_hat, acts_sparse


def standardize_from_train(train_scores, *other_scores):
    # Fit normalization on train residuals only, then apply the same transform to
    # val/test. This avoids leaking val/test distribution information into SAE training.
    train = torch.as_tensor(to_numpy(train_scores), dtype=torch.float32)
    mean = train.mean(dim=0, keepdim=True)
    std = train.std(dim=0, keepdim=True).clamp_min(1e-6)
    standardized = [(train - mean) / std]
    for scores in other_scores:
        x = torch.as_tensor(to_numpy(scores), dtype=torch.float32)
        standardized.append((x - mean) / std)
    return standardized, mean, std


def train_residual_sae(train_scores, cfg):
    # Unsupervised SAE training on train residual representations.
    torch.manual_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X = torch.as_tensor(to_numpy(train_scores), dtype=torch.float32)
    input_dim = X.shape[1]
    dict_size = int(cfg["dict_size_multiplier"] * input_dim)

    model = ResidualBatchTopKSAE(input_dim=input_dim, dict_size=dict_size, top_k=cfg["top_k"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]))

    history = []
    n = X.shape[0]
    batch_size = cfg["batch_size"]

    for epoch in range(cfg["n_epochs"]):
        # Shuffle train residuals each epoch. This is still unsupervised: only X is used.
        perm = torch.randperm(n)
        epoch_loss = 0.0
        # Track mean L0 "activation" (number of nonzero features) as a diagnostic, even though sparsity is really controlled by BatchTopK.
        epoch_l0 = 0.0
        seen = 0
        model.train()

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb = X[idx].to(device)
            # x_hat is the reconstructed residual, acts are the sparse SAE features. Both have shape batch_size x dict_size.
            x_hat, acts = model(xb)

            # Reconstruction loss encourages SAE features to preserve residual information.
            # l1_coeff is currently zero because BatchTopK already enforces sparsity.
            l2 = (x_hat - xb).pow(2).mean()
            l1 = cfg["l1_coeff"] * acts.abs().sum(dim=1).mean()
            loss = l2 + l1

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            opt.step()
            model.renorm_decoder_weights()

            bsz = xb.shape[0]
            epoch_loss += loss.item() * bsz
            epoch_l0 += (acts > 0).float().sum(dim=1).mean().item() * bsz
            seen += bsz

        history.append({
            "epoch": epoch + 1,
            "loss": epoch_loss / seen,
            "mean_l0": epoch_l0 / seen,
        })

        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == cfg["n_epochs"]:
            print(f"epoch {epoch + 1:03d} | loss {history[-1]['loss']:.5f} | mean_l0 {history[-1]['mean_l0']:.2f}")

    return model, pd.DataFrame(history)


@torch.no_grad()
def sae_feature_activations(model, scores, batch_size=4096):
    # Run trained SAE on a split and return sparse feature activations.
    # Shape: n_examples x n_sae_features.
    device = next(model.parameters()).device
    X = torch.as_tensor(to_numpy(scores), dtype=torch.float32)
    acts = []
    model.eval()
    for start in range(0, X.shape[0], batch_size):
        xb = X[start:start + batch_size].to(device)
        _, batch_acts = model(xb)
        acts.append(batch_acts.cpu())
    return torch.cat(acts, dim=0).numpy()


def filter_live_sae_features(train_acts, val_acts, test_acts, min_active_rate=1e-4):
    # Drop dead/near-dead features based on train activations only, then apply the
    # same feature mask to val/test.
    train_active_rate = (train_acts > 0).mean(axis=0)
    live_mask = train_active_rate >= min_active_rate
    live_feature_ids = np.flatnonzero(live_mask)
    return (
        train_acts[:, live_mask],
        val_acts[:, live_mask],
        test_acts[:, live_mask],
        live_feature_ids,
        train_active_rate[live_mask],
    )


def best_binary_threshold(y_true, score):
    # Choose a validation threshold for accuracy/F1. AUC itself is threshold-free,
    # but accuracy/F1 need a cutoff.
    y_true = to_numpy(y_true).astype(int)
    score = to_numpy(score).astype(float)
    candidates = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 99)))
    if len(candidates) == 0:
        return 0.0, np.nan, np.nan

    best_thr = float(candidates[0])
    best_acc = -np.inf
    best_f1 = np.nan
    for thr in candidates:
        pred = (score >= thr).astype(int)
        acc = accuracy_score(y_true, pred)
        if acc > best_acc:
            best_thr = float(thr)
            best_acc = float(acc)
            best_f1 = float(f1_score(y_true, pred, zero_division=0))
    return best_thr, best_acc, best_f1


def match_hidden_to_sae_features(feature_acts, hidden_residuals):
    # Validation-time matching: for each true hidden concept, find the SAE feature
    # whose activation best separates h=1 from h=0 by orientation-free AUC.
    # This uses hidden labels only for evaluation/model selection, not SAE training.
    auc, directions = residual_hidden_auc_matrix(feature_acts, hidden_residuals)
    A = to_numpy(feature_acts).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []

    for h in range(auc.shape[1]):
        f = int(np.nanargmax(auc[:, h]))
        score = A[:, f]
        if directions[f, h] == "negative":
            score = -score
        threshold, val_accuracy, val_f1 = best_binary_threshold(H[:, h], score)
        rows.append({
            "hidden_idx": h,
            "sae_feature_idx": f,
            "auc": float(auc[f, h]),
            "direction": directions[f, h],
            "threshold": threshold,
            "val_accuracy_at_threshold": val_accuracy,
            "val_f1_at_threshold": val_f1,
        })
    return pd.DataFrame(rows), auc, directions


def evaluate_fixed_sae_matches(feature_acts, hidden_residuals, matches):
    # Test-time evaluation: use the validation-chosen SAE feature, orientation,
    # and threshold, then evaluate on held-out test hidden labels.
    A = to_numpy(feature_acts).astype(float)
    H = to_numpy(hidden_residuals).astype(int)
    rows = []

    for row in matches.itertuples(index=False):
        h = int(row.hidden_idx)
        f = int(row.sae_feature_idx)
        score = A[:, f]
        if row.direction == "negative":
            score = -score

        pred = (score >= float(row.threshold)).astype(int)
        rows.append({
            "hidden_idx": h,
            "sae_feature_idx": f,
            "direction": row.direction,
            "threshold": float(row.threshold),
            "sae_auc": roc_auc_score(H[:, h], score),
            "sae_accuracy_at_threshold": accuracy_score(H[:, h], pred),
            "sae_f1_at_threshold": f1_score(H[:, h], pred, zero_division=0),
        })

    return pd.DataFrame(rows)


In [383]:
# -----------------------------
# Train one SAE and compute activations
# -----------------------------
# Standardize residual means using train statistics only. The SAE sees only
# train residual vectors during training; val/test are transformed later for evaluation.
(sae_inputs, sae_train_mean, sae_train_std) = standardize_from_train(
    splits["train"]["res_mu"],
    splits["val"]["res_mu"],
    splits["test"]["res_mu"],
)
X_train_sae, X_val_sae, X_test_sae = sae_inputs

# Train the SAE unsupervised on train residual channel means.
sae_model, sae_history = train_residual_sae(X_train_sae, SAE_CONFIG)
display(sae_history.tail())

# Apply the trained SAE to each split. These activations are the candidate
# discovered residual concepts. Hidden labels are still not used here.
train_sae_acts = sae_feature_activations(sae_model, X_train_sae, SAE_CONFIG["batch_size"])
val_sae_acts = sae_feature_activations(sae_model, X_val_sae, SAE_CONFIG["batch_size"])
test_sae_acts = sae_feature_activations(sae_model, X_test_sae, SAE_CONFIG["batch_size"])

# Remove dead features based on train activity. Keep track of the original SAE
# dictionary ids so evaluation tables can report interpretable feature ids.
(
    train_sae_acts_live,
    val_sae_acts_live,
    test_sae_acts_live,
    live_sae_feature_ids,
    live_sae_active_rate,
) = filter_live_sae_features(
    train_sae_acts,
    val_sae_acts,
    test_sae_acts,
    min_active_rate=SAE_CONFIG["min_train_active_rate"],
)

print(f"SAE dictionary size: {train_sae_acts.shape[1]}")
print(f"Live SAE features: {len(live_sae_feature_ids)}")
print(f"Mean live feature active rate: {live_sae_active_rate.mean():.4f}")


epoch 001 | loss 0.40796 | mean_l0 4.00
epoch 010 | loss 0.03348 | mean_l0 4.00
epoch 020 | loss 0.01859 | mean_l0 4.00
epoch 030 | loss 0.01453 | mean_l0 4.00
epoch 040 | loss 0.01216 | mean_l0 4.00
epoch 050 | loss 0.01117 | mean_l0 4.00


,epoch,loss,mean_l0
45,46,0.011481,4.0
46,47,0.011402,4.0
47,48,0.011322,4.0
48,49,0.011246,4.0
49,50,0.011172,4.0


SAE dictionary size: 40
Live SAE features: 37
Mean live feature active rate: 0.1081


In [384]:
# -----------------------------
# Evaluate SAE features against true hidden concepts
# -----------------------------
# Validation chooses which SAE feature corresponds to each hidden concept.
# This mirrors the raw-axis protocol and avoids choosing matches on the test set.
sae_matches_val, sae_auc_val, sae_directions_val = match_hidden_to_sae_features(
    val_sae_acts_live,
    splits["val"]["hidden_residuals"],
)

# Test evaluates the validation-chosen matches. This is the held-out SAE recovery score.
sae_test_eval = evaluate_fixed_sae_matches(
    test_sae_acts_live,
    splits["test"]["hidden_residuals"],
    sae_matches_val,
)

# Map compact live-feature indices back to original SAE dictionary indices.
sae_test_eval["sae_dict_feature_idx"] = live_sae_feature_ids[sae_test_eval["sae_feature_idx"].values]

# Attach ground-truth task weights only for analysis/ranking. This does not feed
# back into SAE training.
sae_test_eval = attach_relevance(sae_test_eval, relevance).sort_values("rank_abs_w")

print("SAE feature recovery ranked by abs(w_hid)")
display(sae_test_eval[[
    "hidden_idx",
    "sae_dict_feature_idx",
    "sae_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "sae_accuracy_at_threshold",
    "sae_f1_at_threshold",
]])

# Summarize whether SAE recovers the most task-weighted hidden concepts.
print("SAE recovery among top-k hidden concepts by abs(w_hid)")
display(summarize_recovery_by_relevance(sae_test_eval, "sae_auc", "abs_w_hid"))

# Compare three levels of evidence:
# - raw_axis: one original residual coordinate per hidden concept
# - distributed_probe: supervised linear probe, showing information availability
# - SAE: unsupervised sparse features, our concept-discovery attempt
comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(sae_test_eval[["hidden_idx", "sae_auc", "sae_dict_feature_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Raw residual axis vs distributed probe vs SAE")
display(comparison)

comparison_summary = pd.DataFrame({
    "method": ["raw_axis", "distributed_probe", "sae"],
    "top1_mean_auc": [
        comparison.head(1)["axis_auc"].mean(),
        comparison.head(1)["distributed_auc"].mean(),
        comparison.head(1)["sae_auc"].mean(),
    ],
    "top3_mean_auc": [
        comparison.head(3)["axis_auc"].mean(),
        comparison.head(3)["distributed_auc"].mean(),
        comparison.head(3)["sae_auc"].mean(),
    ],
    "top5_mean_auc": [
        comparison.head(5)["axis_auc"].mean(),
        comparison.head(5)["distributed_auc"].mean(),
        comparison.head(5)["sae_auc"].mean(),
    ],
    "spearman_abs_w_auc": [
        comparison[["abs_w_hid", "axis_auc"]].corr(method="spearman").iloc[0, 1],
        comparison[["abs_w_hid", "distributed_auc"]].corr(method="spearman").iloc[0, 1],
        comparison[["abs_w_hid", "sae_auc"]].corr(method="spearman").iloc[0, 1],
    ],
})
display(comparison_summary)


SAE feature recovery ranked by abs(w_hid)


,hidden_idx,sae_dict_feature_idx,sae_auc,w_hid,abs_w_hid,rank_abs_w,sae_accuracy_at_threshold,sae_f1_at_threshold
18,18,13,0.669363,-0.705692,0.705692,1,0.6511,0.666156
2,2,9,0.696964,-0.648612,0.648612,2,0.6660,0.649895
9,9,9,0.636561,0.230210,0.230210,3,0.6174,0.628832
19,19,9,0.533166,-0.150507,0.150507,4,0.5292,0.452431
7,7,13,0.583750,0.075182,0.075182,5,0.5717,0.610601
4,4,13,0.607377,0.000000,0.000000,6,0.5942,0.610631
5,5,13,0.619002,0.000000,0.000000,6,0.6011,0.629103
6,6,16,0.601342,0.000000,0.000000,6,0.5910,0.555435
8,8,9,0.542551,0.000000,0.000000,6,0.5363,0.596256
1,1,2,0.514671,0.000000,0.000000,6,0.5161,0.645624


SAE recovery among top-k hidden concepts by abs(w_hid)


,top_k_by,k,mean_sae_auc,max_sae_auc,num_sae_auc_ge_0_7,hidden_indices
0,abs_w_hid,1,0.669363,0.669363,0,[18]
1,abs_w_hid,3,0.667629,0.696964,0,"[18, 2, 9]"
2,abs_w_hid,5,0.623961,0.696964,0,"[18, 2, 9, 19, 7]"
3,abs_w_hid,10,0.585516,0.696964,0,"[18, 2, 9, 19, 7, 12, 3, 17, 16, 15]"


Raw residual axis vs distributed probe vs SAE


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,sae_auc,sae_dict_feature_idx
0,18,0.700518,0.705692,1,0.713136,0.669363,13
1,2,0.732040,0.648612,2,0.746453,0.696964,9
2,9,0.671554,0.230210,3,0.685063,0.636561,9
3,19,0.544257,0.150507,4,0.571436,0.533166,9
4,7,0.601763,0.075182,5,0.618079,0.583750,13
17,17,0.570079,0.000000,6,0.583489,0.555883,13
16,16,0.572152,0.000000,6,0.589729,0.558584,36
15,15,0.522905,0.000000,6,0.594516,0.525833,11
14,14,0.533031,0.000000,6,0.573579,0.528668,9
13,13,0.627357,0.000000,6,0.647779,0.609722,13


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc,spearman_abs_w_auc
0,raw_axis,0.700518,0.701371,0.650027,0.569182
1,distributed_probe,0.713136,0.714884,0.666833,0.480247
2,sae,0.669363,0.667629,0.623961,0.533608


## SAE Hyperparameter Tuning

The first SAE run can collapse several hidden concepts onto one broad SAE feature. This grid searches over dictionary size, sparsity `top_k`, epochs, and seed, then ranks configs by recovery of the highest `abs(w_hid)` hidden concepts.


In [385]:
# -----------------------------
# SAE hyperparameter tuning
# -----------------------------
# Starter grid. Expand this once the mechanics look sensible.
# Full suggested grid:
#   dict_size_multiplier: [4, 8, 12]
#   top_k: [1, 2, 4, 8]
#   seed: [0, 1, 2]
SAE_TUNING_GRID = [
    {"dict_size_multiplier": d, "top_k": k, "seed": seed}
    for d in [4, 8]
    for k in [1, 2, 4]
    for seed in [0, 1]
]

SAE_TUNING_BASE_CONFIG = {
    **SAE_CONFIG,
    "n_epochs": 75,
}


def ensure_sae_inputs_available():
    """Create standardized SAE inputs if the single-SAE cell has not been run."""
    global X_train_sae, X_val_sae, X_test_sae, sae_train_mean, sae_train_std

    if all(name in globals() for name in ["X_train_sae", "X_val_sae", "X_test_sae"]):
        return X_train_sae, X_val_sae, X_test_sae

    (sae_inputs, sae_train_mean, sae_train_std) = standardize_from_train(
        splits["train"]["res_mu"],
        splits["val"]["res_mu"],
        splits["test"]["res_mu"],
    )
    X_train_sae, X_val_sae, X_test_sae = sae_inputs
    return X_train_sae, X_val_sae, X_test_sae


def evaluate_sae_config(cfg):
    """
    Train one SAE config and evaluate discovered SAE features.

    The SAE is trained without hidden labels. Hidden labels are used only to
    choose feature/hidden matches on validation and evaluate fixed matches on test.
    """
    X_train, X_val, X_test = ensure_sae_inputs_available()

    model, history = train_residual_sae(X_train, cfg)

    train_acts = sae_feature_activations(model, X_train, cfg["batch_size"])
    val_acts = sae_feature_activations(model, X_val, cfg["batch_size"])
    test_acts = sae_feature_activations(model, X_test, cfg["batch_size"])

    # Filter to "live" SAE features that are active on at least a minimum fraction of train examples.
    # Remove dead SAE features that are never active
    train_live, val_live, test_live, live_ids, live_active_rate = filter_live_sae_features(
        train_acts,
        val_acts,
        test_acts,
        min_active_rate=cfg["min_train_active_rate"],
    )

    matches_val, _, _ = match_hidden_to_sae_features(
        val_live,
        splits["val"]["hidden_residuals"],
    )
    test_eval_cfg = evaluate_fixed_sae_matches(
        test_live,
        splits["test"]["hidden_residuals"],
        matches_val,
    )
    test_eval_cfg["sae_dict_feature_idx"] = live_ids[test_eval_cfg["sae_feature_idx"].values]
    test_eval_cfg = attach_relevance(test_eval_cfg, relevance).sort_values("rank_abs_w")

    top1 = test_eval_cfg.head(1)
    top3 = test_eval_cfg.head(3)
    top5 = test_eval_cfg.head(5)
    top10 = test_eval_cfg.head(10)

    matched_top5 = top5["sae_dict_feature_idx"].tolist()
    unique_top5 = len(set(matched_top5))

    summary = {
        "dict_size_multiplier": cfg["dict_size_multiplier"],
        "dict_size": int(cfg["dict_size_multiplier"] * to_numpy(splits["train"]["res_mu"]).shape[1]),
        "top_k": cfg["top_k"],
        "seed": cfg["seed"],
        "n_epochs": cfg["n_epochs"],
        "final_loss": float(history["loss"].iloc[-1]),
        "live_features": int(len(live_ids)),
        "mean_live_active_rate": float(live_active_rate.mean()) if len(live_active_rate) else np.nan,
        "top1_mean_sae_auc": float(top1["sae_auc"].mean()),
        "top3_mean_sae_auc": float(top3["sae_auc"].mean()),
        "top5_mean_sae_auc": float(top5["sae_auc"].mean()),
        "top10_mean_sae_auc": float(top10["sae_auc"].mean()),
        "top5_unique_matched_features": unique_top5,
        "top5_feature_ids": matched_top5,
        "spearman_abs_w_sae_auc": float(test_eval_cfg[["abs_w_hid", "sae_auc"]].corr(method="spearman").iloc[0, 1]),
    }

    return summary, test_eval_cfg, history


# Store one summary row per config, plus detailed per-hidden-concept tables.
sae_tuning_summaries = []
sae_tuning_details = {}
sae_tuning_histories = {}

for i, overrides in enumerate(SAE_TUNING_GRID, start=1):
    cfg = {**SAE_TUNING_BASE_CONFIG, **overrides}
    key = f"d{cfg['dict_size_multiplier']}_k{cfg['top_k']}_seed{cfg['seed']}_ep{cfg['n_epochs']}"
    print("=" * 80)
    print(f"[{i}/{len(SAE_TUNING_GRID)}] {key}")

    # Train/evaluate this config. Hidden labels are used only inside validation/test matching.
    summary, detail, history = evaluate_sae_config(cfg)
    sae_tuning_summaries.append(summary)
    sae_tuning_details[key] = detail
    sae_tuning_histories[key] = history

# Rank configs first by top-5 recovery of the highest-weight hidden concepts,
# then prefer less feature collapse when AUC is tied.
sae_tuning_results = pd.DataFrame(sae_tuning_summaries).sort_values(
    ["top5_mean_sae_auc", "top5_unique_matched_features", "top3_mean_sae_auc"],
    ascending=[False, False, False],
)

print("SAE tuning results ranked by top-5 task-relevant recovery")
display(sae_tuning_results)

# Inspect the best config in detail and compare it to raw-axis/probe baselines.
best_row = sae_tuning_results.iloc[0]
best_sae_key = f"d{int(best_row['dict_size_multiplier'])}_k{int(best_row['top_k'])}_seed{int(best_row['seed'])}_ep{int(best_row['n_epochs'])}"
print("Best SAE config key:", best_sae_key)
print("Best SAE config detail ranked by abs(w_hid)")
display(sae_tuning_details[best_sae_key][[
    "hidden_idx",
    "sae_dict_feature_idx",
    "sae_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "sae_accuracy_at_threshold",
    "sae_f1_at_threshold",
]])

best_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(sae_tuning_details[best_sae_key][["hidden_idx", "sae_auc", "sae_dict_feature_idx"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

print("Best tuned SAE vs raw axis vs distributed probe")
display(best_comparison)

tuned_comparison_summary = pd.DataFrame({
    "method": ["raw_axis", "distributed_probe", "best_tuned_sae"],
    "top1_mean_auc": [
        best_comparison.head(1)["axis_auc"].mean(),
        best_comparison.head(1)["distributed_auc"].mean(),
        best_comparison.head(1)["sae_auc"].mean(),
    ],
    "top3_mean_auc": [
        best_comparison.head(3)["axis_auc"].mean(),
        best_comparison.head(3)["distributed_auc"].mean(),
        best_comparison.head(3)["sae_auc"].mean(),
    ],
    "top5_mean_auc": [
        best_comparison.head(5)["axis_auc"].mean(),
        best_comparison.head(5)["distributed_auc"].mean(),
        best_comparison.head(5)["sae_auc"].mean(),
    ],
})
display(tuned_comparison_summary)


[1/12] d4_k1_seed0_ep75
epoch 001 | loss 0.54430 | mean_l0 1.00
epoch 010 | loss 0.18330 | mean_l0 1.00
epoch 020 | loss 0.07360 | mean_l0 1.00
epoch 030 | loss 0.03472 | mean_l0 1.00
epoch 040 | loss 0.02806 | mean_l0 1.00
epoch 050 | loss 0.02422 | mean_l0 1.00
epoch 060 | loss 0.02118 | mean_l0 1.00
epoch 070 | loss 0.01993 | mean_l0 1.00
epoch 075 | loss 0.01981 | mean_l0 1.00
[2/12] d4_k1_seed1_ep75
epoch 001 | loss 0.60522 | mean_l0 1.00
epoch 010 | loss 0.17653 | mean_l0 1.00
epoch 020 | loss 0.05706 | mean_l0 1.00
epoch 030 | loss 0.03982 | mean_l0 1.00
epoch 040 | loss 0.03472 | mean_l0 1.00
epoch 050 | loss 0.03074 | mean_l0 1.00
epoch 060 | loss 0.02662 | mean_l0 1.00
epoch 070 | loss 0.02263 | mean_l0 1.00
epoch 075 | loss 0.02078 | mean_l0 1.00
[3/12] d4_k2_seed0_ep75
epoch 001 | loss 0.44688 | mean_l0 2.00
epoch 010 | loss 0.06130 | mean_l0 2.00
epoch 020 | loss 0.03340 | mean_l0 2.00
epoch 030 | loss 0.02716 | mean_l0 2.00
epoch 040 | loss 0.02330 | mean_l0 2.00
epoch 05

,dict_size_multiplier,dict_size,top_k,seed,n_epochs,final_loss,live_features,mean_live_active_rate,top1_mean_sae_auc,top3_mean_sae_auc,top5_mean_sae_auc,top10_mean_sae_auc,top5_unique_matched_features,top5_feature_ids,spearman_abs_w_sae_auc
1,4,40,1,1,75,0.020783,21,0.047600,0.678382,0.674196,0.629095,0.602657,1,"[12, 12, 12, 12, 12]",0.533608
2,4,40,2,0,75,0.015925,34,0.058820,0.677206,0.672204,0.627615,0.603490,1,"[13, 13, 13, 13, 13]",0.533608
11,8,80,4,1,75,0.009145,55,0.072713,0.674729,0.670490,0.626719,0.602670,3,"[67, 8, 35, 35, 67]",0.533608
10,8,80,4,0,75,0.010578,57,0.070166,0.673195,0.669763,0.625965,0.602747,3,"[58, 32, 58, 58, 12]",0.517798
5,4,40,4,1,75,0.010030,39,0.102564,0.670996,0.670222,0.625821,0.603351,2,"[12, 24, 12, 24, 12]",0.533608
8,8,80,2,0,75,0.015228,45,0.044431,0.671142,0.670031,0.625436,0.601483,2,"[12, 32, 32, 32, 12]",0.533608
0,4,40,1,0,75,0.019806,21,0.047603,0.673374,0.668021,0.624756,0.599282,2,"[13, 39, 13, 13, 13]",0.533608
3,4,40,2,1,75,0.015916,30,0.066659,0.672939,0.669437,0.624738,0.600834,3,"[12, 18, 12, 24, 12]",0.533608
7,8,80,1,1,75,0.020130,16,0.062467,0.668847,0.669490,0.624239,0.597759,2,"[8, 8, 8, 8, 67]",0.533608
4,4,40,4,0,75,0.009536,38,0.105261,0.670321,0.667397,0.623606,0.600532,3,"[13, 36, 9, 36, 13]",0.533608


Best SAE config key: d4_k1_seed1_ep75
Best SAE config detail ranked by abs(w_hid)


,hidden_idx,sae_dict_feature_idx,sae_auc,w_hid,abs_w_hid,rank_abs_w,sae_accuracy_at_threshold,sae_f1_at_threshold
18,18,12,0.678382,-0.705692,0.705692,1,0.6509,0.674134
2,2,12,0.696891,-0.648612,0.648612,2,0.6677,0.662125
9,9,12,0.647315,0.230210,0.230210,3,0.6191,0.622385
19,19,12,0.536108,-0.150507,0.150507,4,0.5185,0.635255
7,7,12,0.586779,0.075182,0.075182,5,0.5717,0.605653
4,4,12,0.613014,0.000000,0.000000,6,0.5952,0.610320
5,5,12,0.626705,0.000000,0.000000,6,0.6036,0.628073
6,6,12,0.607939,0.000000,0.000000,6,0.5889,0.561400
8,8,18,0.540064,0.000000,0.000000,6,0.5336,0.593232
1,1,12,0.493379,0.000000,0.000000,6,0.5013,0.621997


Best tuned SAE vs raw axis vs distributed probe


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,sae_auc,sae_dict_feature_idx
0,18,0.700518,0.705692,1,0.713136,0.678382,12
1,2,0.732040,0.648612,2,0.746453,0.696891,12
2,9,0.671554,0.230210,3,0.685063,0.647315,12
3,19,0.544257,0.150507,4,0.571436,0.536108,12
4,7,0.601763,0.075182,5,0.618079,0.586779,12
17,17,0.570079,0.000000,6,0.583489,0.558161,12
16,16,0.572152,0.000000,6,0.589729,0.560744,12
15,15,0.522905,0.000000,6,0.594516,0.517961,12
14,14,0.533031,0.000000,6,0.573579,0.524050,18
13,13,0.627357,0.000000,6,0.647779,0.615675,12


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701371,0.650027
1,distributed_probe,0.713136,0.714884,0.666833
2,best_tuned_sae,0.678382,0.674196,0.629095


## Summary of Recovery Approaches

This cell collects the top-1/top-3/top-5 hidden-concept recovery AUCs for every approach that has been run in the notebook. Rows are added only when the corresponding result objects exist.


In [386]:
# -----------------------------
# Summary of all hidden-concept recovery approaches
# -----------------------------
# Each row summarizes recovery of the hidden concepts ranked by abs(w_hid).
# top1/top3/top5 mean AUC therefore answers: among the most task-relevant
# hidden concepts, how well did this method recover them?
#
# This cell is intentionally defensive: it only includes methods whose result
# tables already exist in the notebook state. You can rerun it after running more
# sections and the table will automatically expand.

summary_rows = []


def add_topk_auc_summary(method_name, df, score_col):
    """Append top-1/top-3/top-5 mean AUC for a method, if its table exists."""
    if df is None or score_col not in df.columns:
        return

    # Defensively keep one row per hidden concept. Some upstream sweeps can
    # produce repeated effective top_m values when requested top_m exceeds the
    # number of available directions; duplicates would otherwise make top-3
    # mean AUC count the same hidden concept multiple times.
    ordered = df.sort_values(["rank_abs_w", score_col], ascending=[True, False]).drop_duplicates("hidden_idx")
    summary_rows.append({
        "method": method_name,
        "top1_mean_auc": ordered.head(1)[score_col].mean(),
        "top3_mean_auc": ordered.head(3)[score_col].mean(),
        "top5_mean_auc": ordered.head(5)[score_col].mean(),
    })


# Raw residual axis baseline: validation chooses one residual axis per hidden concept.
if "recovery_eval" in globals():
    add_topk_auc_summary("raw_axis", recovery_eval, "axis_auc")

# Dense supervised hidden probe: upper-bound style diagnostic for linear recoverability.
if "probe_recovery_eval" in globals():
    add_topk_auc_summary("distributed_probe", probe_recovery_eval, "distributed_auc")

# Strict top-k residual probes: supervised diagnostic of how many residual dimensions are needed.
if "topk_probe_recovery_eval" in globals():
    for k in [1, 2, 3, 5, 10, 20, 50]:
        col = f"top{k}_auc"
        if col in topk_probe_recovery_eval.columns:
            add_topk_auc_summary(f"strict_top{k}_probe", topk_probe_recovery_eval, col)

# Sparse L1 probe: supervised diagnostic between raw axes and dense probes.
if "sparse_probe_recovery_eval" in globals():
    add_topk_auc_summary("sparse_l1_probe", sparse_probe_recovery_eval, "sparse_auc")

# Single SAE run.
if "sae_test_eval" in globals():
    add_topk_auc_summary("sae", sae_test_eval, "sae_auc")

# Best tuned SAE run.
if "best_sae_detail" in globals():
    add_topk_auc_summary("best_tuned_sae", best_sae_detail, "sae_auc")
elif "sae_tuning_results" in globals() and "sae_tuning_details" in globals():
    best_row = sae_tuning_results.iloc[0]
    best_key = f"d{int(best_row['dict_size_multiplier'])}_k{int(best_row['top_k'])}_seed{int(best_row['seed'])}_ep{int(best_row['n_epochs'])}"
    if best_key in sae_tuning_details:
        add_topk_auc_summary("best_tuned_sae", sae_tuning_details[best_key], "sae_auc")


# Concept-conditioned task-guided residual direction discovery.
if "best_conditioned_task_guided_detail" in globals():
    add_topk_auc_summary("concept_conditioned_task_guided", best_conditioned_task_guided_detail, "discovery_auc")
elif "conditioned_task_guided_summary" in globals() and "conditioned_task_guided_recovery_eval" in globals():
    best_conditioned = conditioned_task_guided_summary.iloc[0]
    detail = conditioned_task_guided_recovery_eval[
        (conditioned_task_guided_recovery_eval["generator"] == best_conditioned["generator"])
        & (
            conditioned_task_guided_recovery_eval["top_m_task_selected_directions"]
            == best_conditioned["top_m_task_selected_directions"]
        )
    ]
    add_topk_auc_summary("concept_conditioned_task_guided", detail, "discovery_auc")

# Task-guided residual direction discovery.
if "best_task_guided_detail" in globals():
    add_topk_auc_summary("task_guided_discovery", best_task_guided_detail, "discovery_auc")
elif "task_guided_summary" in globals() and "task_guided_recovery_eval" in globals():
    best_task_guided = task_guided_summary.iloc[0]
    detail = task_guided_recovery_eval[
        (task_guided_recovery_eval["generator"] == best_task_guided["generator"])
        & (task_guided_recovery_eval["top_m_task_selected_directions"] == best_task_guided["top_m_task_selected_directions"])
    ]
    add_topk_auc_summary("task_guided_discovery", detail, "discovery_auc")

approach_summary = pd.DataFrame(summary_rows)

# Keep a readable order when those methods are present.
method_order = [
    "raw_axis",
    "distributed_probe",
    "strict_top1_probe",
    "strict_top2_probe",
    "strict_top3_probe",
    "strict_top5_probe",
    "strict_top10_probe",
    "strict_top20_probe",
    "strict_top50_probe",
    "sparse_l1_probe",
    "sae",
    "best_tuned_sae",
    "task_guided_discovery",
    "concept_conditioned_task_guided",
]
if not approach_summary.empty:
    approach_summary["_order"] = approach_summary["method"].map({m: i for i, m in enumerate(method_order)}).fillna(len(method_order))
    approach_summary = approach_summary.sort_values(["_order", "method"]).drop(columns="_order").reset_index(drop=True)

print("Summary of hidden-concept recovery approaches")
display(approach_summary)


Summary of hidden-concept recovery approaches


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701371,0.650027
1,distributed_probe,0.713136,0.714884,0.666833
2,strict_top1_probe,0.699438,0.700305,0.647875
3,strict_top2_probe,0.701194,0.702993,0.657283
4,strict_top3_probe,0.707575,0.706209,0.659800
5,strict_top5_probe,0.712213,0.712206,0.664496
6,strict_top10_probe,0.713136,0.714884,0.666833
7,sparse_l1_probe,0.705682,0.709124,0.662074
8,sae,0.669363,0.667629,0.623961
9,best_tuned_sae,0.678382,0.674196,0.629095


## Interpretation Guide

- Strong validation-chosen test AUCs mean the residual channel itself contains axis-aligned discovered hidden concepts.
- If distributed probe AUC is much higher than axis-aligned AUC, hidden information is present but entangled across residual dimensions.
- If only task-relevant hidden residuals are recovered, that is expected: the residual channel is trained to preserve task-relevant missing information, not necessarily every hidden factor.
- The next step after this baseline is SAE discovery on `train/res_mu.pt`, with feature activations evaluated on val/test.


- Task relevance is ranked by `abs(w_hid)`, the absolute ground-truth hidden-task coefficient. This directly answers whether the residual channel recovers the hidden concepts the synthetic task intended to matter most.
- Signal/prevalence statistics are descriptive diagnostics for visibility or empirical variation; they are not used as the main task-importance metric.

- SAE improves over raw axis discovery if `sae_auc` is higher than `axis_auc` for the top-weight hidden concepts. The distributed probe remains an upper-bound style diagnostic: if probe AUC is high but SAE AUC is low, the information is present but the chosen SAE configuration did not find a useful sparse basis.

- SAE tuning should be judged by top-k `sae_auc` among the highest `abs(w_hid)` hidden concepts, plus feature-collapse diagnostics such as unique matched SAE features in the top-5.

- For hard datasets, hidden task-score recovery (`hidden_residual_signal @ w_hid`) is often more meaningful than binary hidden-concept recovery, because the generator uses continuous residual signal in the label score.

- L1 sparse probes diagnose whether hidden concepts are recoverable from a small supervised linear combination of residual dimensions. If sparse probes approach dense probes, the representation is sparse but rotated; if only dense probes work, it is distributed.
